# BDC 2026 — SigLIP Embedding Probe (Head Architecture Search, v3 patch-max)

Klasifikasi sampah 3 kelas (`Recyclable`, `Electronic`, `Organic`) untuk Big Data Challenge Satria Data 2026, di atas embedding SigLIP v1 + v2 yang dibekukan (frozen backbone), dengan MLP head yang dicari lewat Optuna.

**Model yang jadi juara di notebook ini:** `CE - Full Data (head normal)`, macro F1 test **0.9951**. Konfigurasi lengkap arsitektur dan hyperparameter model ini diekspor otomatis ke `configs/config.json` di akhir sel "Head Normal + Full Data" di bawah, dan juga sudah tersedia siap pakai di repo ini.

## Cara menjalankan di device lain

Notebook ini **portable**, satu-satunya hal yang perlu diganti adalah `ROOT_DIR`, semua path lain (manifest, cache embedding, folder model, ground truth) diturunkan otomatis darinya.

```bash
# opsi 1, lewat environment variable sebelum start Jupyter
export BDC_ROOT_DIR=/path/ke/folder/BDC
jupyter lab

# opsi 2, di sel paling atas Colab
%env BDC_ROOT_DIR=/content/drive/MyDrive/BDC
```

Kalau `BDC_ROOT_DIR` tidak diset, notebook otomatis pakai default Google Drive kalau terdeteksi Colab, atau `./data/BDC` relatif ke lokasi notebook kalau dijalankan lokal.

## Struktur data yang diharapkan di `ROOT_DIR`

```
BDC/
├── Preprocessing_Data/
│   ├── train_manifest.csv
│   └── val_manifest.csv
├── test/                              # folder gambar test
├── gt_updated_no_ambigu.csv           # ground truth test (untuk evaluasi)
├── EDA/
│   ├── label_audit_rekap.csv          # hasil audit cosine-distance/Tomek
│   └── cache_embedding_siglip2_patchmax/   # dibuat otomatis oleh notebook
├── Train_model/Best_model/v3_head_search_siglip2_patchmax/
│   └── optuna_trials_mlp_probe_v3_*.csv    # histori trial Optuna sesi sebelumnya
```



## Reproducibility

- `SEED = 40`, di-set lewat `set_seed()` (Python `random`, NumPy, PyTorch CPU/CUDA, plus `cudnn.deterministic=True`) di sel konfigurasi paling awal.
- Environment persis dicatat di `requirements.txt` pada root repo.


In [ ]:
# ============================================================
# CEK ENVIRONMENT & DEPENDENCY
# ============================================================
# Sel opsional, mempercepat debugging kalau ada versi library yang beda
# antar device (Colab vs lokal vs Kaggle). Tidak mengubah apapun, cuma
# print info.

import sys, platform

def cek_dependency():
    info = {"python": sys.version.split()[0], "platform": platform.platform()}
    for nama_paket in ["torch", "transformers", "numpy", "pandas", "sklearn", "PIL", "optuna"]:
        try:
            mod = __import__(nama_paket if nama_paket != "PIL" else "PIL")
            info[nama_paket] = getattr(mod, "__version__", "tidak diketahui")
        except ImportError:
            info[nama_paket] = "TIDAK TERINSTALL"
    for k, v in info.items():
        print(f"{k:<14} {v}")

cek_dependency()


In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE (HANYA DI GOOGLE COLAB)
# ============================================================
# Sel ini otomatis dilewati kalau notebook tidak dijalankan di Colab
# (misal di Kaggle, JupyterLab lokal, atau server GPU sendiri).
# Path aktual dataset/model diatur terpusat di sel konfigurasi
# berikutnya lewat environment variable BDC_ROOT_DIR, BUKAN di sini.

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUNNING_ON_COLAB = True
except ImportError:
    RUNNING_ON_COLAB = False
    print('Bukan lingkungan Google Colab, sel mount Drive dilewati.')


# SigLIP Embedding Probe, Versi 3 (Head Architecture Search)
Big Data Challenge Satria Data 2026

Notebook ini dibangun dari nol untuk menggantikan notebook sebelumnya yang sudah penuh histori eksperimen (termasuk percobaan partial fine tuning 4 layer terakhir SigLIP yang berat, sekitar satu jam per epoch).

Yang dipertahankan persis sama seperti notebook sebelumnya:

1. Path dan struktur folder di Google Drive
2. Pipeline ekstraksi embedding SigLIP (checkpoint aman, resume aman, N_AUG untuk train, TTA untuk test)
3. Optuna untuk mencari hyperparameter terbaik
4. GroupKFold K-Fold ensemble (5 fold, foto asli dan augmentasinya selalu satu fold)
5. Blend antara model K-Fold dan model full-data

Yang berubah, arsitektur MLP head sekarang punya dua opsi tambahan yang ikut dicari oleh Optuna sejak trial pertama, bukan ditempel belakangan setelah training selesai:

1. Aktivasi, GELU atau SiLU
2. Normalisasi, tanpa norm atau RMSNorm setelah tiap Linear
3. Jumlah layer, satu layer atau dua layer (Two Stage MLP, bottleneck dari dimensi embedding SigLIP 1152 ke hidden_dim, lalu dari hidden_dim ke hidden_dim2 yang lebih kecil lagi)

Karena semua training di sini bekerja di atas embedding yang sudah dicache (bukan melatih ulang backbone SigLIP), satu epoch cuma butuh hitungan detik, bukan satu jam seperti partial fine tuning kemarin.


---

 pembersihan label train/val otomatis dari `label_audit_rekap.csv` (hasil audit cosine-distance/Tomek), TANPA ekstraksi ulang embedding (cuma label & bobot per-baris yang berubah), lalu tuning bobot manual (seperti v8) dilanjutkan dengan pencarian Optuna di sekitar bobot terbaik itu sekaligus arsitektur head, sebelum masuk ke K-Fold + full-data seperti biasa.

In [ ]:
import os
import gc
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, Dataset
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report

# ============================================================
# KONFIGURASI, SEMUA DIATUR DI SINI
# ============================================================

# ROOT_DIR adalah SATU-SATUNYA hal yang perlu diganti untuk menjalankan
# notebook ini di device/environment manapun (Colab, Kaggle, lokal,
# server GPU sendiri). Semua path lain di notebook ini (manifest, cache
# embedding, folder model, ground truth) diturunkan dari ROOT_DIR, JADI
# TIDAK PERLU diedit satu-satu.
#
# Urutan resolusi ROOT_DIR:
#   1. Environment variable BDC_ROOT_DIR (paling direkomendasikan,
#      contoh: export BDC_ROOT_DIR=/workspace/BDC sebelum menjalankan
#      Jupyter, atau %env BDC_ROOT_DIR=... di sel Colab paling atas).
#   2. Kalau tidak diset dan sedang di Colab -> default path Google Drive.
#   3. Kalau tidak diset dan bukan di Colab -> default "./data/BDC" relatif
#      terhadap direktori kerja notebook ini (cocok untuk clone repo
#      GitHub lalu taruh data di situ).
ROOT_DIR = os.environ.get(
    "BDC_ROOT_DIR",
    "/content/drive/MyDrive/BDC" if RUNNING_ON_COLAB else "./data/BDC",
)
ROOT_DIR = os.path.abspath(ROOT_DIR)

TRAIN_MANIFEST_PATH = f"{ROOT_DIR}/Preprocessing_Data/train_manifest.csv"
VAL_MANIFEST_PATH = f"{ROOT_DIR}/Preprocessing_Data/val_manifest.csv"
TEST_DIR_DRIVE = f"{ROOT_DIR}/test"

KOLOM_PATH_MANIFEST = "file_path"
KOLOM_LABEL_MANIFEST = "label"
# SIGLIP_MODEL_NAME diganti ke SigLIP 2 atas saran mentor. CACHE_DIR
# sengaja dibikin ikut nama modelnya (siglip vs siglip2), supaya cache
# embedding dari dua versi model TIDAK saling ketuker/ketimpa, dan
# kalau nanti mau balik ke SigLIP v1 buat dibandingkan, cache lamanya
# masih utuh, tinggal ganti SIGLIP_MODEL_NAME lagi.
SIGLIP_MODEL_NAME = "google/siglip2-so400m-patch14-384"
# Ditambah suffix "_patchmax" supaya cache embedding versi lama (cuma
# pooled/global, 1152 dim) TIDAK ketimpuk/ketuker sama cache versi baru
# (pooled + patch-max, 2304 dim). Kalau nama cache dibiarkan sama, cell
# ekstraksi bakal diam-diam "berhasil" memuat cache lama yang dimensinya
# beda maksud, tanpa error, cuma hasilnya bukan yang kita mau.
TAG_VERSI_SIGLIP = SIGLIP_MODEL_NAME.split("/")[-1].split("-")[0] + "_patchmax"  # "siglip2_patchmax"

CACHE_DIR = f"{ROOT_DIR}/EDA/cache_embedding_{TAG_VERSI_SIGLIP}"
MODEL_SAVE_DIR = f"{ROOT_DIR}/Train_model/Best_model/v3_head_search_{TAG_VERSI_SIGLIP}"

N_AUG = 2
N_TTA = 5

BATCH_SIZE_EKSTRAKSI = 64

# Latency baca Google Drive itu per request, jadi menambah worker
# paralel langsung mempercepat throughput baca tanpa perlu memindah
# data ke disk lokal. Kalau runtime Colab yang dipakai ternyata cuma
# kasih jatah CPU sedikit, turunkan lagi angka ini.
NUM_WORKERS_EKSTRAKSI = 8
PREFETCH_FACTOR_EKSTRAKSI = 2

PATIENCE_EARLY_STOPPING = 8
USE_FP16 = True

# Batas sisi terpanjang gambar sebelum di-downsize, mencegah RAM habis
# untuk gambar beresolusi sangat besar. SigLIP toh akan resize ke
# 384px secara internal, jadi menahan resolusi penuh di RAM sebelum
# itu cuma buang buang memori.
MAX_SISI_GAMBAR = 1024

CHECKPOINT_SETIAP_N_GAMBAR = 2000

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

CACHE_TRAIN = f"{CACHE_DIR}/embedding_train_naug2.npz"
CACHE_VAL = f"{CACHE_DIR}/embedding_val.npz"
CACHE_TEST_CLEAN = f"{CACHE_DIR}/embedding_test_clean.npz"
CACHE_TEST_TTA = f"{CACHE_DIR}/embedding_test_tta5.npz"

LABEL_MAP = {"Recyclable": 0, "Electronic": 1, "Organic": 2}
IDX_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
DAFTAR_LABEL_URUT = sorted(LABEL_MAP, key=lambda x: LABEL_MAP[x])

# ============================================================
# SARAN MENTOR, WEIGHTEDRANDOMSAMPLER DAN BOBOT MANUAL
# ============================================================
# Urutan tiga angka di bawah selalu mengikuti LABEL_MAP, yaitu
# [Recyclable, Electronic, Organic]. GUNAKAN_WEIGHTED_RANDOM_SAMPLER
# mengaktifkan sampling berbobot per epoch (sampel dari kelas dengan
# bobot lebih besar lebih sering terambil, dengan penggantian),
# menggantikan shuffle acak polos. Ini terpisah dari BOBOT_MANUAL_CE
# yang dipakai di CrossEntropyLoss/FocalLoss (bobot per kelas di
# fungsi loss, bukan di sampling data).
GUNAKAN_WEIGHTED_RANDOM_SAMPLER = True

# Recyclable dinaikkan (paling banyak salahnya), Organic diturunkan
# sesuai saran mentor. Electronic dibiarkan di angka netral 1.0.
BOBOT_MANUAL_SAMPLER = [2.0, 1.0, 0.7]

# Dipakai kalau skema_class_weight="manual" saat memanggil training.
BOBOT_MANUAL_CE = [3.5, 1.0, 1.15]

# ============================================================
# RANDOM SEED, agar hasil training selalu reproducible
# ============================================================

SEED = 40

def set_seed(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print(f"Device {DEVICE}")
    print(f"ROOT_DIR {ROOT_DIR}")
    print(f"NUM_WORKERS_EKSTRAKSI dipakai, {NUM_WORKERS_EKSTRAKSI}")

# ============================================================
# Remap Path Manifest dari Windows Lokal ke Path Colab atau Drive
# ============================================================

PATH_LAMA_WINDOWS = r"/content/drive/MyDrive/Satria-Data_IdentifikasiJenisSampah"
PATH_BARU_COLAB = ROOT_DIR
PERLU_REMAP_PATH = True

def remap_path_manifest_jika_perlu(manifest_path, kolom_path):
    if not PERLU_REMAP_PATH:
        return
    df_manifest = pd.read_csv(manifest_path)
    contoh_path = str(df_manifest[kolom_path].iloc[0])
    if PATH_LAMA_WINDOWS.lower() not in contoh_path.lower():
        print(f"Path pada {manifest_path} sudah tidak mengandung path Windows lama, remap dilewati")
        return
    print(f"Path Windows lokal terdeteksi pada {manifest_path}, melakukan remap ke {PATH_BARU_COLAB}")
    df_manifest[kolom_path] = (
        df_manifest[kolom_path]
        .str.replace(PATH_LAMA_WINDOWS, PATH_BARU_COLAB, regex=False, case=False)
        .str.replace("\\", "/", regex=False)
    )
    df_manifest.to_csv(manifest_path, index=False)
    print(f"Remap path selesai, file {manifest_path} sudah diperbarui")

remap_path_manifest_jika_perlu(TRAIN_MANIFEST_PATH, KOLOM_PATH_MANIFEST)
remap_path_manifest_jika_perlu(VAL_MANIFEST_PATH, KOLOM_PATH_MANIFEST)


In [ ]:
# ============================================================
# Utilitas, Baca Gambar Aman Terhadap Resolusi Sangat Besar
# ============================================================

def baca_gambar_aman(path_gambar, ukuran_fallback=384):
    try:
        with Image.open(path_gambar) as img:
            lebar, tinggi = img.size
            if max(lebar, tinggi) > MAX_SISI_GAMBAR:
                rasio = MAX_SISI_GAMBAR / max(lebar, tinggi)
                ukuran_baru = (int(lebar * rasio), int(tinggi * rasio))
                img = img.resize(ukuran_baru, Image.BILINEAR)
            return img.convert("RGB")
    except Exception:
        return Image.new("RGB", (ukuran_fallback, ukuran_fallback), color=(0, 0, 0))


# ============================================================
# Utilitas, Checkpoint dan Resume untuk Proses Ekstraksi
# ============================================================

def muat_checkpoint_jika_ada(cache_path, punya_label):
    checkpoint_path = cache_path.replace(".npz", "_checkpoint.npz")
    if not os.path.exists(checkpoint_path):
        return checkpoint_path, [], ([] if punya_label else None), 0
    print(f"Checkpoint sementara ditemukan di {checkpoint_path}, melanjutkan dari sana...")
    data = np.load(checkpoint_path, allow_pickle=True)
    daftar_embedding = [data["embedding"]]
    daftar_label = list(data["label"]) if punya_label else None
    jumlah_selesai = int(data["jumlah_selesai"])
    print(f"Melanjutkan dari gambar ke-{jumlah_selesai}")
    return checkpoint_path, daftar_embedding, daftar_label, jumlah_selesai


def simpan_checkpoint(checkpoint_path, daftar_embedding, daftar_label, jumlah_selesai):
    embedding_gabung = np.concatenate(daftar_embedding, axis=0)
    if daftar_label is not None:
        np.savez(
            checkpoint_path, embedding=embedding_gabung,
            label=np.array(daftar_label), jumlah_selesai=jumlah_selesai,
        )
    else:
        np.savez(checkpoint_path, embedding=embedding_gabung, jumlah_selesai=jumlah_selesai)


def hapus_checkpoint_jika_ada(checkpoint_path):
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)


In [ ]:
# ============================================================
# Augmentasi untuk Ekstraksi Embedding Train dan TTA Test
# ============================================================

def dapatkan_augmentasi_train():
    return transforms.Compose([
        transforms.RandomResizedCrop(384, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        transforms.RandomGrayscale(p=0.2),
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.0))
        ], p=0.3),
        transforms.RandomRotation(degrees=15),
    ])


def dapatkan_augmentasi_tta(indeks_view):
    if indeks_view == 0:
        return transforms.Compose([transforms.Resize((384, 384))])
    return transforms.Compose([
        transforms.Resize((420, 420)),
        transforms.RandomCrop(384),
        transforms.RandomHorizontalFlip(p=0.5 if indeks_view % 2 == 0 else 0.0),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
    ])


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


In [ ]:
# ============================================================
# Dataset untuk Ekstraksi
#
# Membaca file gambar, menerapkan augmentasi kalau perlu, DAN
# menjalankan image processor SigLIP (resize, normalize, ke tensor),
# semuanya di proses worker terpisah lewat DataLoader, supaya tumpang
# tindih dengan komputasi GPU batch sebelumnya.
# ============================================================

class DatasetEkstraksi(Dataset):
    def __init__(self, daftar_path, daftar_label=None, transform=None, image_processor=None):
        self.daftar_path = daftar_path
        self.daftar_label = daftar_label
        self.transform = transform
        self.image_processor = image_processor

    def __len__(self):
        return len(self.daftar_path)

    def __getitem__(self, idx):
        img = baca_gambar_aman(self.daftar_path[idx])
        if self.transform:
            img = self.transform(img)
        pixel_values = self.image_processor(images=img, return_tensors="pt")["pixel_values"][0]
        label = self.daftar_label[idx] if self.daftar_label is not None else -1
        return pixel_values, label


def collate_tensor(batch):
    pixel_values = torch.stack([item[0] for item in batch], dim=0)
    daftar_label = [item[1] for item in batch]
    return pixel_values, daftar_label


def buat_dataloader_ekstraksi(dataset, shuffle=False):
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE_EKSTRAKSI,
        shuffle=shuffle,
        num_workers=NUM_WORKERS_EKSTRAKSI,
        collate_fn=collate_tensor,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(NUM_WORKERS_EKSTRAKSI > 0),
        prefetch_factor=(PREFETCH_FACTOR_EKSTRAKSI if NUM_WORKERS_EKSTRAKSI > 0 else None),
    )


In [ ]:
# ============================================================
# Ekstraksi Tensor Aman Terhadap Perbedaan Versi Transformers
# ============================================================

def ekstrak_tensor_dari_output(out):
    """
    Diperluas: sekarang mengembalikan (pooled, patch_max) alih-alih cuma
    pooled saja.

    `pooled` (pooler_output) adalah representasi hasil attention-pooling
    (MAP head) SigLIP2 di seluruh gambar -- ini ringkasan GLOBAL, sama
    persis seperti sebelumnya.

    `patch_max` adalah hasil MAX pooling (bukan rata-rata) di seluruh
    token patch mentah (`last_hidden_state`, sebelum masuk MAP head).
    Alasannya: kalau ciri pembeda kelas (misal noda/kontaminasi organik
    di kemasan recyclable) cuma ada di SEBAGIAN KECIL area gambar, MAP
    head yang bersifat pooling-global bisa "melarutkan" sinyal itu ke
    mayoritas piksel lain. Max pooling per-dimensi di seluruh patch
    justru menonjolkan patch yang paling ekstrem responsnya untuk tiap
    channel fitur -- kalau ADA satu patch yang kuat menunjuk ke arah
    kelas tertentu, sinyalnya tetap kebawa, tidak ketutup rata-rata.
    Ini pendekatan yang sama filosofinya dengan Multiple Instance
    Learning (MIL): label gambar ditentukan oleh keberadaan region
    diskriminatif, bukan cuma rata-rata tampilan globalnya.

    Kalau `out` ternyata cuma tensor polos atau tidak punya
    `last_hidden_state` (fallback versi transformers lama), patch_max
    dikembalikan None dan kita otomatis balik ke perilaku lama
    (pooled saja) di `hitung_embedding_batch_aman`.
    """
    if torch.is_tensor(out):
        return out, None
    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        pooled = out.pooler_output
    elif hasattr(out, "last_hidden_state"):
        pooled = out.last_hidden_state[:, 0, :]
    else:
        raise TypeError(f"Tidak tahu cara mengambil tensor dari tipe, {type(out)}")

    patch_max = None
    if hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
        # [B, num_patch, dim] -> max di dimensi patch -> [B, dim]
        patch_max = out.last_hidden_state.max(dim=1).values

    return pooled, patch_max


def muat_siglip():
    # AutoModel/AutoProcessor dipakai (bukan SiglipModel/SiglipProcessor
    # yang di-hardcode) supaya otomatis kedeteksi class yang benar,
    # SigLIP v1 pakai SiglipModel, SigLIP 2 pakai Siglip2Model,
    # class-nya beda walau API-nya (get_image_features, dst) sama.
    from transformers import AutoModel, AutoProcessor
    print(f"Memuat {SIGLIP_MODEL_NAME}")
    processor = AutoProcessor.from_pretrained(SIGLIP_MODEL_NAME)
    dtype = torch.float16 if (USE_FP16 and DEVICE.type == "cuda") else torch.float32
    model = AutoModel.from_pretrained(SIGLIP_MODEL_NAME, torch_dtype=dtype)
    model.eval()
    for param in model.parameters():
        param.requires_grad = False
    model.to(DEVICE)
    print(f"Vision encoder dibekukan ({type(model).__name__})")
    return model, processor


def hitung_embedding_batch_aman(model, batch_pixel_values, batch_size_minimum=1):
    try:
        with torch.no_grad():
            pixel_values = batch_pixel_values.to(DEVICE)
            if USE_FP16 and DEVICE.type == "cuda":
                pixel_values = pixel_values.half()
            # get_image_features() SigLIP/SigLIP2 mengembalikan langsung
            # output vision_model penuh (BaseModelOutputWithPooling), jadi
            # last_hidden_state (token patch mentah) ikut tersedia gratis
            # di forward pass yang sama, tidak perlu forward dua kali.
            pooled, patch_max = ekstrak_tensor_dari_output(
                model.get_image_features(pixel_values=pixel_values)
            )
            pooled = pooled / pooled.norm(dim=-1, keepdim=True)
            if patch_max is not None:
                # Dinormalisasi TERPISAH dari pooled, supaya skala
                # keduanya sebanding sebelum di-concat (pooled sudah unit
                # norm lewat baris di atas, patch_max belum tentu sama
                # skalanya kalau tidak dinormalisasi sendiri).
                patch_max = patch_max / patch_max.norm(dim=-1, keepdim=True)
                fitur = torch.cat([pooled, patch_max], dim=-1)
            else:
                fitur = pooled
        return fitur.float().cpu().numpy()
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gc.collect()
        if len(batch_pixel_values) <= batch_size_minimum:
            raise RuntimeError(
                f"OOM bahkan pada batch berukuran {len(batch_pixel_values)}, "
                f"turunkan lagi BATCH_SIZE_EKSTRAKSI di konfigurasi"
            )
        print(f"OOM terdeteksi pada batch berukuran {len(batch_pixel_values)}, membagi dua dan mencoba ulang")
        titik_tengah = len(batch_pixel_values) // 2
        hasil_pertama = hitung_embedding_batch_aman(model, batch_pixel_values[:titik_tengah], batch_size_minimum)
        hasil_kedua = hitung_embedding_batch_aman(model, batch_pixel_values[titik_tengah:], batch_size_minimum)
        return np.concatenate([hasil_pertama, hasil_kedua], axis=0)


def ekstrak_embedding_train(df_manifest, model, image_processor, cache_path, n_aug):
    if os.path.exists(cache_path):
        print(f"Cache embedding train ditemukan, memuat dari {cache_path}")
        data = np.load(cache_path, allow_pickle=True)
        return data["embedding"], data["label"]

    print(f"Mengekstrak embedding train dengan n_aug {n_aug}")
    daftar_path_diulang, daftar_label_diulang, daftar_versi = [], [], []
    for _, baris in df_manifest.iterrows():
        for versi in range(n_aug):
            daftar_path_diulang.append(baris[KOLOM_PATH_MANIFEST])
            daftar_label_diulang.append(LABEL_MAP[baris[KOLOM_LABEL_MANIFEST]])
            daftar_versi.append(versi)

    total_semua = len(daftar_path_diulang)
    augmentasi_train = dapatkan_augmentasi_train()

    checkpoint_path, daftar_embedding, daftar_label_hasil, jumlah_selesai = muat_checkpoint_jika_ada(
        cache_path, punya_label=True
    )

    if jumlah_selesai >= total_semua:
        print("Seluruh data train (dengan augmentasi) sudah diproses sebelumnya dari checkpoint.")
    else:
        daftar_path_sisa = daftar_path_diulang[jumlah_selesai:]
        daftar_label_sisa = daftar_label_diulang[jumlah_selesai:]
        daftar_versi_sisa = daftar_versi[jumlah_selesai:]

        class DatasetTrainAug(Dataset):
            def __len__(self):
                return len(daftar_path_sisa)

            def __getitem__(self, idx):
                img = baca_gambar_aman(daftar_path_sisa[idx])
                if daftar_versi_sisa[idx] != 0:
                    img = augmentasi_train(img)
                pixel_values = image_processor(images=img, return_tensors="pt")["pixel_values"][0]
                return pixel_values, daftar_label_sisa[idx]

        dataset = DatasetTrainAug()
        loader = buat_dataloader_ekstraksi(dataset, shuffle=False)

        jumlah_diproses_sejak_checkpoint = 0
        for batch_pixel_values, batch_label in tqdm(
            loader, desc="Ekstraksi train", initial=jumlah_selesai // BATCH_SIZE_EKSTRAKSI,
            total=(total_semua // BATCH_SIZE_EKSTRAKSI) + 1,
        ):
            fitur = hitung_embedding_batch_aman(model, batch_pixel_values)
            daftar_embedding.append(fitur)
            daftar_label_hasil.extend(batch_label)
            jumlah_diproses_sejak_checkpoint += len(batch_label)
            if jumlah_diproses_sejak_checkpoint >= CHECKPOINT_SETIAP_N_GAMBAR:
                simpan_checkpoint(checkpoint_path, daftar_embedding, daftar_label_hasil, len(daftar_label_hasil))
                jumlah_diproses_sejak_checkpoint = 0

    embedding_array = np.concatenate(daftar_embedding, axis=0)
    label_array = np.array(daftar_label_hasil)
    np.savez(cache_path, embedding=embedding_array, label=label_array)
    print(f"Embedding train selesai, bentuk {embedding_array.shape}, disimpan ke {cache_path}")
    hapus_checkpoint_jika_ada(checkpoint_path)
    return embedding_array, label_array


def ekstrak_embedding_val(df_manifest, model, image_processor, cache_path):
    if os.path.exists(cache_path):
        print(f"Cache embedding validasi ditemukan, memuat dari {cache_path}")
        data = np.load(cache_path, allow_pickle=True)
        return data["embedding"], data["label"]

    print("Mengekstrak embedding validasi")
    daftar_path = df_manifest[KOLOM_PATH_MANIFEST].tolist()
    daftar_label = [LABEL_MAP[l] for l in df_manifest[KOLOM_LABEL_MANIFEST].tolist()]
    total_semua = len(daftar_path)

    checkpoint_path, daftar_embedding, daftar_label_hasil, jumlah_selesai = muat_checkpoint_jika_ada(
        cache_path, punya_label=True
    )

    if jumlah_selesai >= total_semua:
        print("Seluruh data validasi sudah diproses sebelumnya dari checkpoint.")
    else:
        dataset = DatasetEkstraksi(
            daftar_path[jumlah_selesai:], daftar_label[jumlah_selesai:],
            transform=None, image_processor=image_processor,
        )
        loader = buat_dataloader_ekstraksi(dataset, shuffle=False)

        jumlah_diproses_sejak_checkpoint = 0
        for batch_pixel_values, batch_label in tqdm(loader, desc="Ekstraksi validasi"):
            fitur = hitung_embedding_batch_aman(model, batch_pixel_values)
            daftar_embedding.append(fitur)
            daftar_label_hasil.extend(batch_label)
            jumlah_diproses_sejak_checkpoint += len(batch_label)
            if jumlah_diproses_sejak_checkpoint >= CHECKPOINT_SETIAP_N_GAMBAR:
                simpan_checkpoint(checkpoint_path, daftar_embedding, daftar_label_hasil, len(daftar_label_hasil))
                jumlah_diproses_sejak_checkpoint = 0

    embedding_array = np.concatenate(daftar_embedding, axis=0)
    label_array = np.array(daftar_label_hasil)
    np.savez(cache_path, embedding=embedding_array, label=label_array)
    print(f"Embedding validasi selesai, bentuk {embedding_array.shape}, disimpan ke {cache_path}")
    hapus_checkpoint_jika_ada(checkpoint_path)
    return embedding_array, label_array


def daftar_file_test_terurut(test_dir):
    file_list = [f for f in os.listdir(test_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    file_list = sorted(file_list, key=lambda x: int(os.path.splitext(x)[0]))
    return file_list


def ekstrak_embedding_test_clean(test_dir, model, image_processor, cache_path):
    if os.path.exists(cache_path):
        print(f"Cache embedding test bersih ditemukan, memuat dari {cache_path}")
        data = np.load(cache_path, allow_pickle=True)
        return data["embedding"], list(data["nama_file"])

    print("Mengekstrak embedding test, versi bersih")
    file_list = daftar_file_test_terurut(test_dir)
    daftar_path = [os.path.join(test_dir, f) for f in file_list]
    total_semua = len(daftar_path)

    checkpoint_path, daftar_embedding, _, jumlah_selesai = muat_checkpoint_jika_ada(
        cache_path, punya_label=False
    )

    if jumlah_selesai >= total_semua:
        print("Seluruh data test (bersih) sudah diproses sebelumnya dari checkpoint.")
    else:
        dataset = DatasetEkstraksi(
            daftar_path[jumlah_selesai:], daftar_label=None,
            transform=None, image_processor=image_processor,
        )
        loader = buat_dataloader_ekstraksi(dataset, shuffle=False)

        jumlah_diproses_sejak_checkpoint = 0
        jumlah_total_terkumpul = jumlah_selesai
        for batch_pixel_values, _ in tqdm(loader, desc="Ekstraksi test bersih"):
            fitur = hitung_embedding_batch_aman(model, batch_pixel_values)
            daftar_embedding.append(fitur)
            jumlah_total_terkumpul += len(fitur)
            jumlah_diproses_sejak_checkpoint += len(fitur)
            if jumlah_diproses_sejak_checkpoint >= CHECKPOINT_SETIAP_N_GAMBAR:
                simpan_checkpoint(checkpoint_path, daftar_embedding, None, jumlah_total_terkumpul)
                jumlah_diproses_sejak_checkpoint = 0

    embedding_array = np.concatenate(daftar_embedding, axis=0)
    np.savez(cache_path, embedding=embedding_array, nama_file=np.array(file_list))
    print(f"Embedding test bersih selesai, bentuk {embedding_array.shape}")
    hapus_checkpoint_jika_ada(checkpoint_path)
    return embedding_array, file_list


def ekstrak_embedding_test_tta(test_dir, model, image_processor, cache_path, n_tta):
    if os.path.exists(cache_path):
        print(f"Cache embedding test TTA ditemukan, memuat dari {cache_path}")
        data = np.load(cache_path, allow_pickle=True)
        return data["embedding"], list(data["nama_file"])

    print(f"Mengekstrak embedding test, {n_tta} view TTA per gambar")
    file_list = daftar_file_test_terurut(test_dir)
    daftar_path = [os.path.join(test_dir, f) for f in file_list]

    embedding_per_view = []
    for indeks_view in range(n_tta):
        cache_path_view = cache_path.replace(".npz", f"_view{indeks_view}.npz")
        if os.path.exists(cache_path_view):
            print(f"Cache TTA view {indeks_view} ditemukan, memuat dari {cache_path_view}")
            embedding_per_view.append(np.load(cache_path_view)["embedding"])
            continue

        augmentasi = dapatkan_augmentasi_tta(indeks_view)
        total_semua = len(daftar_path)

        checkpoint_path, daftar_embedding, _, jumlah_selesai = muat_checkpoint_jika_ada(
            cache_path_view, punya_label=False
        )

        if jumlah_selesai >= total_semua:
            print(f"View TTA {indeks_view} sudah diproses sebelumnya dari checkpoint.")
        else:
            dataset = DatasetEkstraksi(
                daftar_path[jumlah_selesai:], daftar_label=None,
                transform=augmentasi, image_processor=image_processor,
            )
            loader = buat_dataloader_ekstraksi(dataset, shuffle=False)

            jumlah_diproses_sejak_checkpoint = 0
            jumlah_total_terkumpul = jumlah_selesai
            for batch_pixel_values, _ in tqdm(loader, desc=f"TTA view {indeks_view}"):
                fitur = hitung_embedding_batch_aman(model, batch_pixel_values)
                daftar_embedding.append(fitur)
                jumlah_total_terkumpul += len(fitur)
                jumlah_diproses_sejak_checkpoint += len(fitur)
                if jumlah_diproses_sejak_checkpoint >= CHECKPOINT_SETIAP_N_GAMBAR:
                    simpan_checkpoint(checkpoint_path, daftar_embedding, None, jumlah_total_terkumpul)
                    jumlah_diproses_sejak_checkpoint = 0

        embedding_view = np.concatenate(daftar_embedding, axis=0)
        np.savez(cache_path_view, embedding=embedding_view)
        hapus_checkpoint_jika_ada(checkpoint_path)
        embedding_per_view.append(embedding_view)

    embedding_array = np.stack(embedding_per_view, axis=0)
    np.savez(cache_path, embedding=embedding_array, nama_file=np.array(file_list))
    print(f"Embedding test TTA selesai, bentuk {embedding_array.shape}")

    for indeks_view in range(n_tta):
        cache_path_view = cache_path.replace(".npz", f"_view{indeks_view}.npz")
        if os.path.exists(cache_path_view):
            os.remove(cache_path_view)

    return embedding_array, file_list


In [ ]:
# ============================================================
# JALANKAN EKSTRAKSI
# ============================================================
# Kalau cache sudah ada di CACHE_DIR, fungsi di atas otomatis memuat
# dari cache dan tidak mengekstrak ulang, jadi cell ini aman dijalankan
# ulang kapan saja.

df_train = pd.read_csv(TRAIN_MANIFEST_PATH)
df_val = pd.read_csv(VAL_MANIFEST_PATH)

model_siglip, processor_siglip = muat_siglip()
image_processor = processor_siglip.image_processor

embedding_train, label_train = ekstrak_embedding_train(
    df_train, model_siglip, image_processor, CACHE_TRAIN, N_AUG
)
embedding_val, label_val = ekstrak_embedding_val(
    df_val, model_siglip, image_processor, CACHE_VAL
)
embedding_test_clean, file_list_test = ekstrak_embedding_test_clean(
    TEST_DIR_DRIVE, model_siglip, image_processor, CACHE_TEST_CLEAN
)
embedding_test_tta, _ = ekstrak_embedding_test_tta(
    TEST_DIR_DRIVE, model_siglip, image_processor, CACHE_TEST_TTA, N_TTA
)

dim_embedding = embedding_train.shape[1]
print(f"\nDimensi embedding SigLIP, {dim_embedding}")
print(f"Total baris train setelah augmentasi, {embedding_train.shape[0]}")
print(f"Embedding val {embedding_val.shape}, test clean {embedding_test_clean.shape}, test TTA {embedding_test_tta.shape}")

print("\nMembebaskan VRAM SigLIP setelah seluruh ekstraksi selesai")
del model_siglip
del processor_siglip
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


### Concat Embedding SigLIP v1 + v2 (v2 sekarang sudah pooled+patch-max, 2304 dim)

Embedding SigLIP v2 yang barusan diekstrak (lihat perubahan di fungsi
`hitung_embedding_batch_aman`/`ekstrak_tensor_dari_output` di atas) sekarang
sudah berisi concat [pooled, patch-max] per gambar (1152 + 1152 = 2304 dim),
bukan pooled saja seperti sebelumnya. Cell ini menggabungkan itu dengan
embedding SigLIP v1 yang sudah dicache dari sesi sebelumnya (masih pooled
saja, 1152 dim), jadi dimensi akhirnya 1152 + 2304 = 3456.

Ini butuh cache SigLIP v1 sudah ada duluan di `cache_embedding_siglip` (dari
sesi sebelum kita ganti ke SigLIP2). Kalau belum ada, cell ini bakal kasih
error jelas, bukan diam-diam salah.


In [ ]:
# ============================================================
# LOAD & CONCAT EMBEDDING SIGLIP V1 + V2
# ============================================================

CACHE_DIR_V1 = f"{ROOT_DIR}/EDA/cache_embedding_siglip"

daftar_path_v1 = {
    "train": f"{CACHE_DIR_V1}/embedding_train_naug{N_AUG}.npz",
    "val": f"{CACHE_DIR_V1}/embedding_val.npz",
    "test_clean": f"{CACHE_DIR_V1}/embedding_test_clean.npz",
    "test_tta": f"{CACHE_DIR_V1}/embedding_test_tta{N_TTA}.npz",
}

for nama, path in daftar_path_v1.items():
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Cache SigLIP v1 buat '{nama}' tidak ditemukan di {path}. "
            f"Concat butuh embedding v1 dan v2 dengan N_AUG={N_AUG} dan N_TTA={N_TTA} yang SAMA. "
            f"Kalau cache v1 kamu dulu diekstrak dengan N_AUG/N_TTA berbeda, sesuaikan dulu "
            f"variabel N_AUG/N_TTA di config sebelum jalankan cell ini, atau ekstrak ulang v1."
        )

print("Memuat cache SigLIP v1...")
data_train_v1 = np.load(daftar_path_v1["train"], allow_pickle=True)
embedding_train_v1, label_train_v1 = data_train_v1["embedding"], data_train_v1["label"]

data_val_v1 = np.load(daftar_path_v1["val"], allow_pickle=True)
embedding_val_v1, label_val_v1 = data_val_v1["embedding"], data_val_v1["label"]

data_test_clean_v1 = np.load(daftar_path_v1["test_clean"], allow_pickle=True)
embedding_test_clean_v1, file_list_test_v1 = data_test_clean_v1["embedding"], list(data_test_clean_v1["nama_file"])

data_test_tta_v1 = np.load(daftar_path_v1["test_tta"], allow_pickle=True)
embedding_test_tta_v1 = data_test_tta_v1["embedding"]

# ---- Validasi, pastikan urutan/jumlah baris v1 dan v2 (yang sekarang di memori) match ----
assert embedding_train_v1.shape[0] == embedding_train.shape[0], (
    f"Jumlah baris embedding_train v1 ({embedding_train_v1.shape[0]}) dan v2 ({embedding_train.shape[0]}) beda, "
    f"kemungkinan diekstrak dengan N_AUG berbeda, cek ulang."
)
assert np.array_equal(label_train_v1, label_train), "Label train v1 dan v2 tidak sama urutannya, cek ulang manifest"
assert file_list_test_v1 == file_list_test, "Urutan file test v1 dan v2 tidak sama, cek ulang folder test"
assert embedding_test_tta_v1.shape[0] == embedding_test_tta.shape[0], "Jumlah view TTA v1 dan v2 beda (N_TTA berbeda)"

print(f"Dimensi embedding v1, {embedding_train_v1.shape[1]}, v2, {embedding_train.shape[1]}")

# ---- Concat di dimensi fitur (axis paling belakang) ----
embedding_train = np.concatenate([embedding_train_v1, embedding_train], axis=-1)
embedding_val = np.concatenate([embedding_val_v1, embedding_val], axis=-1)
embedding_test_clean = np.concatenate([embedding_test_clean_v1, embedding_test_clean], axis=-1)
embedding_test_tta = np.concatenate([embedding_test_tta_v1, embedding_test_tta], axis=-1)

dim_embedding = embedding_train.shape[1]

print(f"\nSetelah concat, dimensi embedding jadi {dim_embedding} (harusnya 2x lipat dari salah satu versi)")
print(f"embedding_train {embedding_train.shape}, embedding_val {embedding_val.shape}")
print(f"embedding_test_clean {embedding_test_clean.shape}, embedding_test_tta {embedding_test_tta.shape}")

del embedding_train_v1, embedding_val_v1, embedding_test_clean_v1, embedding_test_tta_v1
del data_train_v1, data_val_v1, data_test_clean_v1, data_test_tta_v1

### [DIGANTI] Fitur Handcrafted Material-Level -> Patch-Max Pooling

Percobaan sebelumnya (histogram HSV + GLCM + rasio gelap/reflektif,
di-concat sebagai fitur tambahan 30 dim) sudah dicoba dan **hasilnya
turun** (F1 test 0.9957 -> 0.9930), dan ID yang tadinya ambigu (1136,
1399, 565, dst) malah makin pede salah (gap probabilitas naik, bukan
turun). Diagnosisnya: histogram/tekstur itu dihitung sebagai satu
ringkasan statistik dari SELURUH piksel gambar -- masalahnya sama
persis dengan pooled embedding SigLIP, yaitu sinyal lokal (noda di
sebagian kecil area) ke-rata-rata/dilarutkan oleh mayoritas piksel
background yang bersih.

Pendekatan yang dipakai sekarang (lihat fungsi `ekstrak_tensor_dari_output`
dan `hitung_embedding_batch_aman` di atas): **max pooling di seluruh
token patch mentah SigLIP2** (`last_hidden_state`, sebelum MAP head),
bukan cuma pakai `pooler_output` (yang notabene sudah hasil pooling
attention -- tetap global). Kalau ada satu/beberapa patch yang secara
tajam menunjuk ke ciri kelas tertentu, max pooling menonjolkan itu,
tidak ketutup rata-rata seperti pooled/handcrafted kemarin. Ini "gratis"
dari sisi komputasi karena `last_hidden_state` sudah tersedia di forward
pass yang sama dengan `pooler_output`, tidak perlu ekstraksi/forward
pass tambahan.

Sel-sel ekstraksi fitur handcrafted (histogram HSV, GLCM, rasio
gelap/reflektif) yang sebelumnya ada di sini sudah dihapus dari
notebook ini.


## Arsitektur Head Baru, RMSNorm dan SiLU Jadi Bagian Resmi Pencarian Optuna

Saran dari mentor, daripada melatih ulang beberapa layer terakhir SigLIP (berat, sekitar satu jam per epoch), lebih efisien memperkuat MLP head yang bekerja di atas embedding beku, sambil menambah kapasitas normalisasi dan aktivasinya.

Dua bahan yang ditambahkan ke `MLPProbeClassifier`.

1. RMSNorm, opsional dipasang setelah tiap `nn.Linear`. Menjaga skala aktivasi tetap stabil, terutama saat head dibuat lebih dalam (dua layer).
2. SiLU, sebagai alternatif GELU. Gradiennya lebih mulus dibanding GELU di daerah negatif, sering membantu MLP kecil menemukan minimum yang lebih rata.

Kombinasi paling direkomendasikan, Two Stage MLP (`n_layers=2`) dengan bottleneck dari 1152 ke 512 lalu ke dimensi hidden kedua yang lebih kecil, dipasangkan SiLU dan RMSNorm, plus `Dropout` 0.2 sampai 0.3 sebagai regularizer terhadap sekitar 50 ribu baris data. Nilai persis untuk tiap hyperparameter tetap dicari lewat Optuna, bukan dikunci manual, supaya pencarian tetap adil dibanding opsi satu layer atau kombinasi GELU tanpa norm.


In [ ]:
# ============================================================
# RMSNorm dan MLPProbeClassifier, Versi 3
# ============================================================

class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization (Zhang and Sennrich, 2019).
    Menormalkan skala aktivasi lewat root mean square saja, tanpa
    menggeser mean seperti LayerNorm biasa. Lebih ringan secara
    komputasi, dan sering dipasangkan dengan SiLU di arsitektur modern
    untuk menjaga stabilitas nilai aktivasi selama training.
    """
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.weight


class MLPProbeClassifier(nn.Module):
    """
    MLP head di atas embedding SigLIP beku. Dibanding versi sebelumnya,
    sekarang activation dan norm_type adalah hyperparameter resmi yang
    dicari Optuna sejak trial pertama, bukan ditempel belakangan.

    activation, "gelu" atau "silu"
    norm_type, "none" (tanpa norm, seperti versi sebelumnya) atau
        "rmsnorm" (RMSNorm dipasang setelah tiap Linear, sebelum
        activation)
    n_layers, 1 (satu Linear ke num_classes) atau 2 (Two Stage MLP,
        bottleneck dim_embedding -> hidden_dim -> hidden_dim2 ->
        num_classes)
    """
    def __init__(
        self, dim_embedding, hidden_dim=512, num_classes=3, dropout=0.3,
        activation="silu", norm_type="rmsnorm", n_layers=1, hidden_dim2=None,
    ):
        super().__init__()

        def buat_aktivasi():
            if activation == "gelu":
                return nn.GELU()
            elif activation == "silu":
                return nn.SiLU()
            raise ValueError(f"activation '{activation}' tidak dikenali, pakai 'gelu' atau 'silu'")

        def buat_norm(dim):
            if norm_type == "rmsnorm":
                return RMSNorm(dim)
            elif norm_type == "none":
                return nn.Identity()
            raise ValueError(f"norm_type '{norm_type}' tidak dikenali, pakai 'none' atau 'rmsnorm'")

        if n_layers == 1:
            self.net = nn.Sequential(
                nn.Linear(dim_embedding, hidden_dim),
                buat_norm(hidden_dim),
                buat_aktivasi(),
                nn.Dropout(p=dropout),
                nn.Linear(hidden_dim, num_classes),
            )
        elif n_layers == 2:
            if hidden_dim2 is None:
                raise ValueError("hidden_dim2 wajib diisi kalau n_layers=2")
            self.net = nn.Sequential(
                nn.Linear(dim_embedding, hidden_dim),
                buat_norm(hidden_dim),
                buat_aktivasi(),
                nn.Dropout(p=dropout),
                nn.Linear(hidden_dim, hidden_dim2),
                buat_norm(hidden_dim2),
                buat_aktivasi(),
                nn.Dropout(p=dropout),
                nn.Linear(hidden_dim2, num_classes),
            )
        else:
            raise ValueError(f"n_layers={n_layers} tidak didukung, cuma 1 atau 2")

    def forward(self, x):
        return self.net(x)


In [ ]:
# ============================================================
# Class Weight, Focal Loss, dan Dispatcher Criterion
# ============================================================

def hitung_class_weight(label_array, num_classes=3):
    total = len(label_array)
    counts = [int((label_array == i).sum()) for i in range(num_classes)]
    weights = [total / (num_classes * c) if c > 0 else 0.0 for c in counts]
    for i, nama in enumerate(DAFTAR_LABEL_URUT):
        print(f"Kelas {nama}, jumlah {counts[i]}, bobot {weights[i]:.4f}")
    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)


def hitung_class_weight_effective_number(label_array, beta=0.999):
    kelas_unik = sorted(LABEL_MAP.values())
    jumlah_per_kelas = np.array([int((label_array == k).sum()) for k in kelas_unik])
    effective_num = 1.0 - np.power(beta, jumlah_per_kelas)
    bobot = (1.0 - beta) / effective_num
    bobot = bobot / bobot.sum() * len(kelas_unik)
    for k, w, n in zip(kelas_unik, bobot, jumlah_per_kelas):
        print(f"Kelas {IDX_TO_LABEL[k]}, jumlah {n}, bobot effective-number (beta={beta}) {w:.4f}")
    return torch.tensor(bobot, dtype=torch.float32).to(DEVICE)


def pilih_class_weight(label_array, skema="inverse_freq", beta=0.999, bobot_manual=None):
    if skema == "inverse_freq":
        return hitung_class_weight(label_array)
    elif skema == "effective_number":
        return hitung_class_weight_effective_number(label_array, beta=beta)
    elif skema == "manual":
        if bobot_manual is None:
            raise ValueError("skema_class_weight='manual' butuh bobot_manual diisi, contoh BOBOT_MANUAL_CE")
        for nama, w in zip(DAFTAR_LABEL_URUT, bobot_manual):
            print(f"Kelas {nama}, bobot manual {w:.4f}")
        return torch.tensor(bobot_manual, dtype=torch.float32).to(DEVICE)
    else:
        raise ValueError(f"skema_class_weight '{skema}' tidak dikenali, pakai 'inverse_freq', 'effective_number', atau 'manual'")


# ============================================================
# WeightedRandomSampler Manual (Sesuai Saran Mentor)
#
# Karena training di notebook ini tidak lewat DataLoader (seluruh
# embedding sudah ada di GPU sekaligus, permutasi indeks dibuat
# langsung tiap epoch), WeightedRandomSampler sungguhan dari
# torch.utils.data tidak dipakai secara literal. Sebagai gantinya,
# efek yang sama direplikasi lewat torch.multinomial dengan
# penggantian (with replacement), bobot per sampel diambil dari bobot
# kelasnya. Hasilnya setara, sampel dari kelas berbobot besar (mis.
# Recyclable) lebih sering terambil tiap epoch dibanding shuffle
# acak polos, sedangkan kelas berbobot kecil (mis. Organic) lebih
# jarang terambil.
# ============================================================

def buat_bobot_sampler_per_kelas(bobot_manual_sampler):
    return torch.tensor(bobot_manual_sampler, dtype=torch.float32, device=DEVICE)


def buat_index_epoch(n_baris, y_tensor, gunakan_sampler, bobot_sampler_per_kelas):
    if not gunakan_sampler or bobot_sampler_per_kelas is None:
        return torch.randperm(n_baris, device=DEVICE)
    bobot_per_sampel = bobot_sampler_per_kelas[y_tensor]
    return torch.multinomial(bobot_per_sampel, n_baris, replacement=True)


class FocalLoss(nn.Module):
    """
    Sampel yang gampang diklasifikasi (pt tinggi) kontribusinya ke loss
    diredam lewat faktor (1-pt)^gamma, sementara sampel yang susah
    tetap dikasih bobot penuh. alpha tetap dipakai untuk class weight,
    jadi ini gabungan class weighting dan hard example weighting.

    [BARU, v9] forward() sekarang menerima sample_weight opsional,
    bobot PER BARIS (bukan per kelas) hasil resolusi audit
    cosine-distance (baris DOWNWEIGHT dapat bobot < 1). Kalau None,
    perilakunya identik seperti sebelumnya (mean polos).
    """
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, target, sample_weight=None):
        ce_per_sampel = nn.functional.cross_entropy(
            logits, target, weight=self.alpha,
            label_smoothing=self.label_smoothing, reduction="none",
        )
        pt = torch.exp(-ce_per_sampel)
        focal_per_sampel = ((1 - pt) ** self.gamma) * ce_per_sampel
        if sample_weight is not None:
            return (focal_per_sampel * sample_weight).sum() / sample_weight.sum().clamp_min(1e-8)
        return focal_per_sampel.mean()


class CEPerSampelWrapper(nn.Module):
    """
    [BARU, v9] nn.CrossEntropyLoss bawaan cuma dukung bobot PER KELAS
    lewat argumen weight=. Supaya baris hasil DOWNWEIGHT (bobot PER
    SAMPEL, dari resolusi audit cosine-distance) juga ikut
    diperhitungkan di loss_type="ce", CE dihitung reduction='none'
    dulu baru digabung manual dengan sample_weight -- pola yang sama
    persis seperti FocalLoss di atas, supaya "ce" dan "focal"
    konsisten cara nanganin sample_weight-nya.
    """
    def __init__(self, weight=None, label_smoothing=0.0):
        super().__init__()
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, logits, target, sample_weight=None):
        ce_per_sampel = nn.functional.cross_entropy(
            logits, target, weight=self.weight,
            label_smoothing=self.label_smoothing, reduction="none",
        )
        if sample_weight is not None:
            return (ce_per_sampel * sample_weight).sum() / sample_weight.sum().clamp_min(1e-8)
        return ce_per_sampel.mean()


def buat_criterion(loss_type, class_weight, label_smoothing, focal_gamma=None):
    if loss_type == "ce":
        return CEPerSampelWrapper(weight=class_weight, label_smoothing=label_smoothing)
    elif loss_type == "focal":
        gamma_dipakai = 2.0 if focal_gamma is None else focal_gamma
        return FocalLoss(alpha=class_weight, gamma=gamma_dipakai, label_smoothing=label_smoothing)
    else:
        raise ValueError(f"loss_type '{loss_type}' tidak dikenali, pakai 'ce' atau 'focal'")


In [ ]:
# ============================================================
# EarlyStopping yang Mencatat Epoch Terbaik, dan Fungsi Latih
# GPU Resident (Tanpa DataLoader, Seluruh Embedding di GPU Sekali)
# ============================================================

class EarlyStoppingCatatEpoch:
    def __init__(self, patience=PATIENCE_EARLY_STOPPING, delta=0.0):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.best_epoch = 0
        self.counter = 0
        self.early_stop = False
        self.best_state_dict = None

    def __call__(self, current_score, model, epoch):
        if self.best_score is None or current_score > (self.best_score + self.delta):
            self.best_score = current_score
            self.best_epoch = epoch
            self.counter = 0
            self.best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


def latih_mlp_dengan_earlystop(
    embedding_tr, label_tr, embedding_va, label_va,
    hidden_dim, dropout, lr, weight_decay, label_smoothing,
    batch_size, epochs_maks, patience, dim_in, seed=SEED, verbose=False,
    trial=None, epoch_offset=0, n_layers=1, hidden_dim2=None,
    activation="silu", norm_type="rmsnorm",
    loss_type="ce", focal_gamma=None,
    skema_class_weight="inverse_freq", beta_effective_number=0.999,
    bobot_manual_class_weight=None,
    gunakan_sampler=False, bobot_sampler_manual=None,
    sample_weight_tr=None,  # [BARU, v9] bobot per baris hasil resolusi audit (DOWNWEIGHT < 1)
):
    set_seed(seed)

    X_tr = torch.as_tensor(embedding_tr, dtype=torch.float32, device=DEVICE)
    y_tr = torch.as_tensor(label_tr, dtype=torch.long, device=DEVICE)
    X_va = torch.as_tensor(embedding_va, dtype=torch.float32, device=DEVICE)
    y_va = torch.as_tensor(label_va, dtype=torch.long, device=DEVICE)
    y_va_np = y_va.cpu().numpy()
    sw_tr = (
        torch.as_tensor(sample_weight_tr, dtype=torch.float32, device=DEVICE)
        if sample_weight_tr is not None else None
    )

    class_weight = pilih_class_weight(
        label_tr, skema=skema_class_weight, beta=beta_effective_number,
        bobot_manual=bobot_manual_class_weight,
    )
    model = MLPProbeClassifier(
        dim_in, hidden_dim=hidden_dim, dropout=dropout,
        n_layers=n_layers, hidden_dim2=hidden_dim2,
        activation=activation, norm_type=norm_type,
    ).to(DEVICE)
    criterion = buat_criterion(loss_type, class_weight, label_smoothing, focal_gamma)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    stopper = EarlyStoppingCatatEpoch(patience=patience)

    n_train = X_tr.shape[0]
    bobot_sampler_per_kelas = (
        buat_bobot_sampler_per_kelas(bobot_sampler_manual) if gunakan_sampler else None
    )

    for epoch in range(1, epochs_maks + 1):
        model.train()
        perm = buat_index_epoch(n_train, y_tr, gunakan_sampler, bobot_sampler_per_kelas)
        total_loss = 0.0
        n_batch = 0
        for start in range(0, n_train, batch_size):
            idx = perm[start:start + batch_size]
            optimizer.zero_grad()
            output = model(X_tr[idx])
            loss = criterion(output, y_tr[idx], sw_tr[idx] if sw_tr is not None else None)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batch += 1
        rata_loss_train = total_loss / n_batch

        model.eval()
        with torch.no_grad():
            output_val = model(X_va)
            pred_val = torch.argmax(output_val, dim=1).cpu().numpy()
        macro_f1_val = f1_score(y_va_np, pred_val, average="macro")

        stopper(macro_f1_val, model, epoch)

        if verbose:
            print(f"  epoch {epoch}, loss train {rata_loss_train:.4f}, macro f1 val {macro_f1_val:.4f}")

        if trial is not None:
            trial.report(macro_f1_val, epoch_offset + epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if stopper.early_stop:
            break

    model.load_state_dict(stopper.best_state_dict)
    return model, stopper.best_score, stopper.best_epoch


def prediksi_probs_dengan_tta(model, embedding_clean, embedding_tta):
    model.eval()
    seluruh_probs = []
    with torch.no_grad():
        tensor_clean = torch.tensor(embedding_clean, dtype=torch.float32).to(DEVICE)
        seluruh_probs.append(torch.softmax(model(tensor_clean), dim=1))
        for indeks_view in range(embedding_tta.shape[0]):
            tensor_view = torch.tensor(embedding_tta[indeks_view], dtype=torch.float32).to(DEVICE)
            seluruh_probs.append(torch.softmax(model(tensor_view), dim=1))
    return torch.stack(seluruh_probs, dim=0).mean(dim=0)


def bandingkan_dengan_ground_truth(nama_model, prediksi_idx, file_list_test, path_label_test):
    daftar_id = [int(os.path.splitext(f)[0]) for f in file_list_test]
    df_pred = pd.DataFrame({"id": daftar_id, "predicted": prediksi_idx.astype(int)})

    if not os.path.exists(path_label_test):
        print(f"[{nama_model}] File ground truth tidak ditemukan di {path_label_test}, evaluasi dilewati")
        return None

    df_gt = pd.read_csv(path_label_test, encoding="utf-8-sig", sep=None, engine="python")
    df_gt.columns = [c.strip() for c in df_gt.columns]

    if "id" in df_gt.columns:
        df_gt["id"] = df_gt["id"].astype(str).str.strip()
        df_gt["id"] = pd.to_numeric(df_gt["id"], errors="coerce")
    elif "file" in df_gt.columns:
        df_gt["file"] = df_gt["file"].astype(str).str.strip()
        df_gt["id"] = df_gt["file"].apply(lambda x: os.path.splitext(x)[0])
        df_gt["id"] = pd.to_numeric(df_gt["id"], errors="coerce")
    else:
        print(f"[{nama_model}] Error, ground truth {path_label_test} harus punya kolom 'id' atau 'file'.")
        return None

    if "label" not in df_gt.columns:
        print(f"[{nama_model}] Error, ground truth {path_label_test} tidak punya kolom 'label'.")
        return None

    df_gt["label"] = df_gt["label"].astype(str).str.strip()
    df_gt["label"] = pd.to_numeric(df_gt["label"], errors="coerce")

    baris_valid = df_gt["label"].isin([0, 1, 2]) & df_gt["id"].notna()
    jumlah_dibuang = int((~baris_valid).sum())
    if jumlah_dibuang > 0:
        print(f"[{nama_model}] Peringatan, {jumlah_dibuang} baris di ground truth tidak valid dan dibuang")

    df_gt_bersih = df_gt.loc[baris_valid, ["id", "label"]].copy()
    df_gt_bersih["id"] = df_gt_bersih["id"].astype(int)
    df_gt_bersih["label"] = df_gt_bersih["label"].astype(int)

    df_gabung = pd.merge(df_pred, df_gt_bersih, on="id", how="inner")
    if len(df_gabung) == 0:
        print(f"[{nama_model}] Error, tidak ada id yang cocok antara prediksi dan ground truth")
        return None

    macro_f1 = f1_score(df_gabung["label"], df_gabung["predicted"], average="macro")
    print(f"[{nama_model}] Macro F1 di test (n={len(df_gabung)} sampel), {macro_f1:.4f}")
    return macro_f1



In [ ]:
# ============================================================
# Rekonstruksi Group ID untuk GroupKFold
#
# Optuna TIDAK dijalankan ulang di sesi ini, konfigurasi terbaik
# langsung dibaca dari file CSV histori trial yang sudah tersimpan
# dari sesi sebelumnya (lihat cell berikutnya).
# ============================================================
from sklearn.model_selection import GroupKFold
from datetime import datetime
import glob

assert embedding_train.shape[0] == len(df_train) * N_AUG, (
    "Jumlah baris embedding_train tidak habis dibagi len(df_train) * N_AUG. "
    "Kemungkinan cache dibuat dengan N_AUG yang berbeda dari nilai N_AUG saat ini."
)

groups_train = np.repeat(np.arange(len(df_train)), N_AUG)
groups_val = np.arange(len(df_train), len(df_train) + len(df_val))

embedding_all = np.concatenate([embedding_train, embedding_val], axis=0)
label_all = np.concatenate([label_train, label_val], axis=0)
groups_all = np.concatenate([groups_train, groups_val], axis=0)

print(
    f"Total baris embedding gabungan (train+val), {embedding_all.shape[0]}, "
    f"berasal dari {len(np.unique(groups_all))} gambar unik"
)

TIMESTAMP_RUN = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Timestamp sesi ini, {TIMESTAMP_RUN} (dipakai di semua nama file yang disimpan)")

LABEL_TEST_PATH_BANDING = f"{ROOT_DIR}/gt_updated_no_ambigu.csv"


## [BARU, v9] Pembersihan Label Train dari Hasil Audit Cosine-Distance (Tomek)

`label_audit_rekap.csv` berisi hasil audit manual atas pasangan Tomek dan outlier lingkungan yang ditemukan lewat jarak cosine di embedding SigLIP2 patch-max ini juga (`baris_asli` = `index_baris`, indeks baris di `embedding_all`/`groups_all`, persis definisi yang dipakai di notebook `v15_tomek_visual` waktu audit ini dibuat). Karena embedding yang dipakai sama-sama SigLIP2 patch-max, `baris_asli` di CSV valid dipetakan langsung ke `df_train`/`df_val` notebook ini lewat `groups_all`.

Dari 1423 baris audit, ada 1055 `baris_asli` unik, dan 147 di antaranya (146 kelompok dengan 2 verdict berbeda + 1 kelompok dengan 3 verdict berbeda) py verdict-nya tidak konsisten karena baris yang sama muncul sebagai tetangga di lebih dari satu pasangan Tomek. Sesuai arahan, seluruh 147 kelompok ini dianggap **GAMBAR_SAMA_beda_pendapat** (bukan dicurigai beda gambar), dan diresolusi otomatis, bukan dicek satu-satu, dengan prioritas:

**REMAP > DOWNWEIGHT > KEEP**

Kalau dalam satu kelompok ada lebih dari satu target REMAP yang berbeda (ditemukan cuma di 3 dari 367 kelompok baris_asli terduplikasi, dan ketiganya kebetulan seri 1 lawan 1), dipilih target dengan suara terbanyak; kalau seri, jatuh ke DOWNWEIGHT sebagai pilihan aman (label asli dipertahankan, bobotnya diturunkan saja, tidak ditebak sepihak mau di-REMAP ke mana).

Penting: yang berubah cuma **label** dan **bobot per-baris**, bukan piksel gambarnya. Jadi `embedding_train` / `embedding_val` / `embedding_all` dari cache SigLIP2 patch-max yang sudah ada TETAP DIPAKAI APA ADANYA, tidak perlu ekstraksi ulang. Yang diekstrak ulang cuma kalau daftar gambarnya sendiri berubah (ada yang ditambah/dibuang), bukan kasus di sini.

Baris berstatus DOWNWEIGHT diberi bobot loss `DOWNWEIGHT_FACTOR` (default 0.5, bisa diubah) lewat mekanisme sample-weight baru yang ditambahkan ke `FocalLoss`/CE dan fungsi training di bawah (lihat cell setelah ini).


In [ ]:
# ============================================================
# [BARU, v9] PEMBERSIHAN LABEL TRAIN/VAL DARI AUDIT COSINE-DISTANCE
# ============================================================

LABEL_AUDIT_CSV_PATH = f"{ROOT_DIR}/EDA/label_audit_rekap.csv"
DOWNWEIGHT_FACTOR = 0.5  # bobot loss untuk baris berstatus DOWNWEIGHT, silakan diubah

assert os.path.exists(LABEL_AUDIT_CSV_PATH), (
    f"File audit tidak ditemukan di {LABEL_AUDIT_CSV_PATH}. "
    f"Upload label_audit_rekap.csv ke folder itu di Drive dulu, atau sesuaikan LABEL_AUDIT_CSV_PATH."
)

df_audit = pd.read_csv(LABEL_AUDIT_CSV_PATH)


def urutan_prioritas_action(action):
    if action.startswith("REMAP"):
        return 0
    if action == "DOWNWEIGHT":
        return 1
    return 2  # KEEP


def resolusi_satu_kelompok(g):
    """
    Meresolusi satu kelompok baris_asli (bisa 1 baris atau banyak baris
    kalau baris itu muncul di beberapa pasangan Tomek dengan verdict
    beda-beda) jadi satu keputusan akhir, prioritas REMAP > DOWNWEIGHT >
    KEEP. Kalau REMAP menang tapi ada >1 target berbeda, dipilih suara
    terbanyak; seri -> jatuh ke DOWNWEIGHT (aman, tidak menebak sepihak).
    """
    aksi_unik = g["action"].unique()
    aksi_menang = sorted(aksi_unik, key=urutan_prioritas_action)[0]

    if not aksi_menang.startswith("REMAP"):
        return aksi_menang, None

    semua_target_remap = [a for a in aksi_unik if a.startswith("REMAP")]
    if len(semua_target_remap) == 1:
        label_tujuan = semua_target_remap[0].replace("REMAP ke ", "").strip()
        return "REMAP", label_tujuan

    hitungan = g[g["action"].isin(semua_target_remap)]["action"].value_counts()
    if len(hitungan) > 1 and hitungan.iloc[0] == hitungan.iloc[1]:
        return "DOWNWEIGHT", None  # seri antar target REMAP, jatuh ke DOWNWEIGHT
    label_tujuan = hitungan.idxmax().replace("REMAP ke ", "").strip()
    return "REMAP", label_tujuan


hasil_resolusi = []
for baris_asli_val, g in df_audit.groupby("baris_asli"):
    aksi_final, label_tujuan_final = resolusi_satu_kelompok(g)
    hasil_resolusi.append({
        "baris_asli": int(baris_asli_val),
        "label_asli": g["label_asli"].iloc[0],
        "aksi_final": aksi_final,
        "label_tujuan": label_tujuan_final,
        "n_baris_csv": len(g),
        "n_verdict_beda": g["verdict"].nunique(),
    })

df_resolusi = pd.DataFrame(hasil_resolusi)
n_konflik = int((df_resolusi["n_verdict_beda"] > 1).sum())
print(f"Total baris_asli unik di audit, {len(df_resolusi)}")
print(f"Kelompok dengan verdict konflik (diresolusi otomatis, prioritas REMAP > DOWNWEIGHT > KEEP), {n_konflik}")
print(df_resolusi["aksi_final"].value_counts())
print("\nKontribusi target REMAP,")
print(df_resolusi.loc[df_resolusi["aksi_final"] == "REMAP", "label_tujuan"].value_counts())

# ---- Terapkan resolusi ke label_train dan label_val ----
label_train_bersih = label_train.copy()
sample_weight_train = np.ones(len(label_train), dtype=np.float32)
label_val_bersih = label_val.copy()
sample_weight_val = np.ones(len(label_val), dtype=np.float32)

n_train_group = len(df_train)
jumlah_baris_remap = 0
jumlah_baris_downweight = 0
jumlah_baris_asli_di_luar_jangkauan = 0

for _, r in df_resolusi.iterrows():
    idx_baris = int(r["baris_asli"])
    if idx_baris < 0 or idx_baris >= len(groups_all):
        jumlah_baris_asli_di_luar_jangkauan += 1
        continue
    group_id = int(groups_all[idx_baris])

    if group_id < n_train_group:
        baris_terkena = np.where(groups_train == group_id)[0]
        target_label, target_weight = label_train_bersih, sample_weight_train
    else:
        baris_terkena = np.where(groups_val == group_id)[0]
        target_label, target_weight = label_val_bersih, sample_weight_val

    if len(baris_terkena) == 0:
        continue

    if r["aksi_final"] == "REMAP":
        target_label[baris_terkena] = LABEL_MAP[r["label_tujuan"]]
        jumlah_baris_remap += len(baris_terkena)
    elif r["aksi_final"] == "DOWNWEIGHT":
        target_weight[baris_terkena] = DOWNWEIGHT_FACTOR
        jumlah_baris_downweight += len(baris_terkena)
    # KEEP -> tidak ada perubahan

if jumlah_baris_asli_di_luar_jangkauan > 0:
    print(f"\nPeringatan, {jumlah_baris_asli_di_luar_jangkauan} baris_asli di CSV di luar jangkauan embedding_all saat ini, dilewati")

print(f"\nJumlah baris embedding_train/embedding_val yang label-nya di-REMAP, {jumlah_baris_remap}")
print(f"Jumlah baris yang bobot loss-nya diturunkan jadi {DOWNWEIGHT_FACTOR}, {jumlah_baris_downweight}")

# ---- TIDAK PERLU EKSTRAKSI ULANG EMBEDDING ----
# Yang berubah cuma label dan bobot per baris, bukan piksel gambarnya,
# jadi embedding_train/embedding_val (cache SigLIP2 patch-max yang
# sudah ada) tetap valid dipakai apa adanya.

# ---- Timpa variabel global yang dipakai semua cell training di bawah,
# supaya tuning bobot, K-Fold, dan full-data OTOMATIS memakai versi
# bersih tanpa perlu mengubah cell-cell itu satu per satu ----
label_train, label_val = label_train_bersih, label_val_bersih
label_all = np.concatenate([label_train, label_val], axis=0)
sample_weight_all = np.concatenate([sample_weight_train, sample_weight_val], axis=0)

print("\nDistribusi label_train setelah dibersihkan,")
print(pd.Series(label_train).map(IDX_TO_LABEL).value_counts())

os.makedirs(f"{ROOT_DIR}/EDA", exist_ok=True)
path_resolusi_tersimpan = f"{ROOT_DIR}/EDA/label_resolusi_final_{TAG_VERSI_SIGLIP}_{TIMESTAMP_RUN}.csv"
df_resolusi.to_csv(path_resolusi_tersimpan, index=False)
print(f"\nTabel resolusi lengkap (buat ditelusuri manual kalau perlu) disimpan ke {path_resolusi_tersimpan}")


In [ ]:
# ============================================================
# OPTUNA DILEWATI DI SESI INI
# ============================================================
# Sesuai arahan, Optuna tidak dijalankan ulang untuk mencari
# hyperparameter. Konfigurasi terbaik diambil langsung dari file CSV
# histori trial (optuna_trials_mlp_probe_v3_*.csv) yang sudah
# tersimpan di MODEL_SAVE_DIR dari sesi training sebelumnya, lihat
# cell berikutnya.

print("Optuna dilewati, memuat konfigurasi terbaik dari CSV histori trial sebelumnya")


In [ ]:
# ============================================================
# Ambil Konfigurasi Terbaik dari CSV Histori Trial Sebelumnya
# ============================================================
# Menggantikan cara lama (ambil dari objek `study` hasil study.optimize
# yang baru saja jalan). Sekarang dibaca langsung dari file CSV yang
# sudah tersimpan di MODEL_SAVE_DIR pada sesi Optuna sebelumnya.
# Kalau ada beberapa file (beberapa sesi berbeda), yang dipakai adalah
# yang namanya terakhir secara urutan abjad (biasanya juga yang
# timestamp-nya paling baru, karena format nama pakai %Y%m%d_%H%M%S).

def muat_konfigurasi_terbaik_dari_csv(model_save_dir, loss_type_target):
    daftar_csv = sorted(glob.glob(f"{model_save_dir}/optuna_trials_mlp_probe_v3_*.csv"))
    if not daftar_csv:
        raise FileNotFoundError(
            f"Tidak ada file optuna_trials_mlp_probe_v3_*.csv ditemukan di {model_save_dir}. "
            f"Pastikan sesi Optuna sebelumnya sudah pernah dijalankan dan hasilnya tersimpan di Drive."
        )
    path_csv_terpakai = daftar_csv[-1]
    print(f"Memuat histori trial dari {path_csv_terpakai}")
    df_trials = pd.read_csv(path_csv_terpakai)
    df_selesai_lokal = df_trials[df_trials["state"] == "COMPLETE"].copy()

    df_filter = df_selesai_lokal[df_selesai_lokal["params_loss_type"] == loss_type_target]
    if len(df_filter) == 0:
        print(f"Tidak ada trial loss_type={loss_type_target} yang selesai penuh di CSV ini, dilewati")
        return None

    baris_terbaik = df_filter.loc[df_filter["value"].idxmax()]

    konfigurasi = {}
    for kolom in df_filter.columns:
        if not kolom.startswith("params_"):
            continue
        nama_param = kolom.replace("params_", "")
        nilai = baris_terbaik[kolom]
        if pd.isna(nilai):
            continue
        konfigurasi[nama_param] = nilai

    # Tipe angka bulat perlu dipaksa balik ke int, karena lewat CSV
    # semua kolom numerik otomatis kebaca sebagai float.
    for kolom_int in ["n_layers", "hidden_dim", "hidden_dim2", "batch_size"]:
        if kolom_int in konfigurasi and konfigurasi[kolom_int] is not None:
            konfigurasi[kolom_int] = int(konfigurasi[kolom_int])

    kolom_best_epoch = "user_attrs_best_epoch"
    if kolom_best_epoch not in baris_terbaik or pd.isna(baris_terbaik[kolom_best_epoch]):
        raise KeyError(f"Kolom {kolom_best_epoch} tidak ditemukan di CSV, cek ulang isi filenya")
    konfigurasi["best_epoch"] = int(baris_terbaik[kolom_best_epoch])

    konfigurasi.setdefault("n_layers", 1)
    konfigurasi.setdefault("hidden_dim2", None)
    konfigurasi.setdefault("activation", "gelu")
    konfigurasi.setdefault("norm_type", "none")

    print(f"Trial terbaik untuk loss_type={loss_type_target}, macro F1 validasi {baris_terbaik['value']:.4f}")
    print(f"  {konfigurasi}")
    return konfigurasi


konfigurasi_ce = muat_konfigurasi_terbaik_dari_csv(MODEL_SAVE_DIR, "ce")
konfigurasi_focal = muat_konfigurasi_terbaik_dari_csv(MODEL_SAVE_DIR, "focal")

# ---- Suntikkan saran mentor, WeightedRandomSampler dan bobot manual ----
# Arsitektur (hidden_dim, n_layers, activation, dst) tetap dari hasil
# Optuna di atas, cuma skema class weight dan strategi sampling yang
# diganti manual sesuai arahan mentor.
for konfigurasi in (konfigurasi_ce, konfigurasi_focal):
    if konfigurasi is None:
        continue
    konfigurasi["skema_class_weight"] = "manual"
    konfigurasi["bobot_manual_class_weight"] = BOBOT_MANUAL_CE
    konfigurasi["gunakan_sampler"] = GUNAKAN_WEIGHTED_RANDOM_SAMPLER
    konfigurasi["bobot_sampler_manual"] = BOBOT_MANUAL_SAMPLER

print("\nKonfigurasi terbaik (sudah disuntik bobot manual dan sampler), CrossEntropyLoss,")
print(f"  {konfigurasi_ce}")
print("\nKonfigurasi terbaik (sudah disuntik bobot manual dan sampler), FocalLoss,")
print(f"  {konfigurasi_focal}")


## Tuning Bobot WeightedRandomSampler dan Class Weight

Angka bobot di konfigurasi awal (`[2.0, 1.0, 0.7]` untuk sampler, `[3.5, 1.0, 1.15]` untuk CrossEntropyLoss) dicari lewat pencarian kecil di sekitar titik itu, bukan langsung ditembak. Electronic dikunci di 1.0 sebagai titik acuan (paling kecil bobotnya, sesuai temuan awal), Recyclable dan Organic yang divariasikan.

Prosesnya dua tahap, mirip coordinate descent. Tahap 1 tuning bobot sampler dulu (class weight CE dikunci sementara di titik acuan), tahap 2 baru tuning bobot class weight CE (sampler dikunci di hasil terbaik tahap 1). Tiap kombinasi dievaluasi cepat lewat `latih_mlp_dengan_earlystop` di split train/val asli (bukan K-Fold, supaya satu kombinasi cuma butuh satu training singkat), pakai arsitektur head yang sama seperti hasil Optuna (`konfigurasi_ce`), cuma bobotnya yang beda beda.

Begitu ketemu kombinasi terbaik, `BOBOT_MANUAL_SAMPLER` dan `BOBOT_MANUAL_CE` diperbarui otomatis, dan disuntikkan lagi ke `konfigurasi_ce` / `konfigurasi_focal`. Jadi cell K-Fold, head normal, dan analisis kesalahan di bawahnya otomatis memakai angka hasil tuning ini kalau notebook dijalankan ulang dari atas.

In [ ]:
# ============================================================
# TUNING RINGAN BOBOT SAMPLER DAN CLASS WEIGHT
# ============================================================

def evaluasi_kombinasi_bobot(bobot_ce, bobot_sampler, gunakan_sampler, arsitektur):
    _, skor_val, _ = latih_mlp_dengan_earlystop(
        embedding_train, label_train, embedding_val, label_val,
        hidden_dim=arsitektur["hidden_dim"], dropout=arsitektur["dropout"],
        lr=arsitektur["lr"], weight_decay=arsitektur["weight_decay"],
        label_smoothing=arsitektur["label_smoothing"], batch_size=arsitektur["batch_size"],
        epochs_maks=60, patience=PATIENCE_EARLY_STOPPING, dim_in=dim_embedding, verbose=False,
        n_layers=arsitektur.get("n_layers", 1), hidden_dim2=arsitektur.get("hidden_dim2"),
        activation=arsitektur.get("activation", "silu"), norm_type=arsitektur.get("norm_type", "rmsnorm"),
        loss_type=arsitektur["loss_type"], focal_gamma=arsitektur.get("focal_gamma"),
        skema_class_weight="manual", bobot_manual_class_weight=bobot_ce,
        gunakan_sampler=gunakan_sampler, bobot_sampler_manual=bobot_sampler,
        sample_weight_tr=sample_weight_train,  # [BARU, v9] ikutkan bobot DOWNWEIGHT hasil audit
    )
    return skor_val


assert konfigurasi_ce is not None, "konfigurasi_ce kosong, jalankan cell muat konfigurasi terbaik dulu"
arsitektur_dipakai = konfigurasi_ce

# ---- Tahap 1, tuning bobot WeightedRandomSampler ----
# Class weight CE dikunci sementara di titik acuan awal, biar cuma satu
# hal yang berubah tiap percobaan.
BOBOT_CE_ACUAN_SEMENTARA = [3.5, 1.0, 1.15]

kandidat_sampler_recyclable = [1.7, 2.0, 2.3]
kandidat_sampler_organic = [0.6, 0.7, 0.8]

hasil_tuning_sampler = []
for r in kandidat_sampler_recyclable:
    for o in kandidat_sampler_organic:
        bobot_sampler_dicoba = [r, 1.0, o]
        skor = evaluasi_kombinasi_bobot(BOBOT_CE_ACUAN_SEMENTARA, bobot_sampler_dicoba, True, arsitektur_dipakai)
        hasil_tuning_sampler.append({"recyclable": r, "electronic": 1.0, "organic": o, "macro_f1_val": skor})
        print(f"Sampler [Recyclable {r}, Electronic 1.0, Organic {o}], macro F1 validasi {skor:.4f}")

df_tuning_sampler = pd.DataFrame(hasil_tuning_sampler).sort_values("macro_f1_val", ascending=False).reset_index(drop=True)
print("\nHasil tuning bobot sampler, terurut dari yang terbaik")
print(df_tuning_sampler.to_string(index=False))

baris_terbaik_sampler = df_tuning_sampler.iloc[0]
BOBOT_MANUAL_SAMPLER_HASIL_TUNING = [
    float(baris_terbaik_sampler["recyclable"]), float(baris_terbaik_sampler["electronic"]), float(baris_terbaik_sampler["organic"]),
]
print(f"\nBobot sampler terbaik hasil tuning, {BOBOT_MANUAL_SAMPLER_HASIL_TUNING} (macro F1 validasi {baris_terbaik_sampler['macro_f1_val']:.4f})")

# ---- Tahap 2, tuning bobot class weight CrossEntropyLoss ----
# Bobot sampler dikunci di hasil terbaik tahap 1.
kandidat_ce_recyclable = [3.0, 3.5, 4.0]
kandidat_ce_organic = [1.0, 1.15, 1.3]

hasil_tuning_ce = []
for r in kandidat_ce_recyclable:
    for o in kandidat_ce_organic:
        bobot_ce_dicoba = [r, 1.0, o]
        skor = evaluasi_kombinasi_bobot(bobot_ce_dicoba, BOBOT_MANUAL_SAMPLER_HASIL_TUNING, True, arsitektur_dipakai)
        hasil_tuning_ce.append({"recyclable": r, "electronic": 1.0, "organic": o, "macro_f1_val": skor})
        print(f"CE weight [Recyclable {r}, Electronic 1.0, Organic {o}], macro F1 validasi {skor:.4f}")

df_tuning_ce = pd.DataFrame(hasil_tuning_ce).sort_values("macro_f1_val", ascending=False).reset_index(drop=True)
print("\nHasil tuning class weight CrossEntropyLoss, terurut dari yang terbaik")
print(df_tuning_ce.to_string(index=False))

baris_terbaik_ce = df_tuning_ce.iloc[0]
BOBOT_MANUAL_CE_HASIL_TUNING = [
    float(baris_terbaik_ce["recyclable"]), float(baris_terbaik_ce["electronic"]), float(baris_terbaik_ce["organic"]),
]
print(f"\nBobot class weight CE terbaik hasil tuning, {BOBOT_MANUAL_CE_HASIL_TUNING} (macro F1 validasi {baris_terbaik_ce['macro_f1_val']:.4f})")

# ---- Terapkan hasil tuning ke konfigurasi yang dipakai cell cell berikutnya ----
BOBOT_MANUAL_SAMPLER = BOBOT_MANUAL_SAMPLER_HASIL_TUNING
BOBOT_MANUAL_CE = BOBOT_MANUAL_CE_HASIL_TUNING

for konfigurasi in (konfigurasi_ce, konfigurasi_focal):
    if konfigurasi is None:
        continue
    konfigurasi["bobot_manual_class_weight"] = BOBOT_MANUAL_CE
    konfigurasi["bobot_sampler_manual"] = BOBOT_MANUAL_SAMPLER

print(f"\nBOBOT_MANUAL_SAMPLER diperbarui ke {BOBOT_MANUAL_SAMPLER}")
print(f"BOBOT_MANUAL_CE diperbarui ke {BOBOT_MANUAL_CE}")
print("Cell K-Fold, head normal + full data, dan analisis kesalahan di bawah otomatis memakai bobot hasil tuning ini")


In [ ]:
# ============================================================
# TAHAP 3, PERLUASAN GRID CLASS WEIGHT CE
# ============================================================
# Hasil terbaik tahap 2 kemarin, [3.0, 1.0, 1.3], berada persis di
# ujung rentang yang dicoba (Recyclable di batas bawah 3.0, Organic di
# batas atas 1.3). Itu tanda optimum sebenarnya mungkin masih di luar
# rentang itu, jadi digeser lagi ke arah yang sama, Recyclable makin
# turun, Organic makin naik. Sampler dikunci di hasil tahap 1.

kandidat_ce_recyclable_lanjutan = [2.5, 2.75, 3.0]
kandidat_ce_organic_lanjutan = [1.3, 1.45, 1.6]

hasil_tuning_ce_lanjutan = []
for r in kandidat_ce_recyclable_lanjutan:
    for o in kandidat_ce_organic_lanjutan:
        bobot_ce_dicoba = [r, 1.0, o]
        skor = evaluasi_kombinasi_bobot(bobot_ce_dicoba, BOBOT_MANUAL_SAMPLER_HASIL_TUNING, True, arsitektur_dipakai)
        hasil_tuning_ce_lanjutan.append({"recyclable": r, "electronic": 1.0, "organic": o, "macro_f1_val": skor})
        print(f"CE weight lanjutan [Recyclable {r}, Electronic 1.0, Organic {o}], macro F1 validasi {skor:.4f}")

df_tuning_ce_lanjutan = pd.DataFrame(hasil_tuning_ce_lanjutan).sort_values("macro_f1_val", ascending=False).reset_index(drop=True)
print("\nHasil tuning class weight CE, tahap lanjutan, terurut dari yang terbaik")
print(df_tuning_ce_lanjutan.to_string(index=False))

# ---- Gabungkan dengan hasil tahap 2, ambil yang paling bagus dari semuanya ----
df_tuning_ce_gabungan = pd.concat([df_tuning_ce, df_tuning_ce_lanjutan], ignore_index=True)
df_tuning_ce_gabungan = df_tuning_ce_gabungan.drop_duplicates(subset=["recyclable", "electronic", "organic"])
df_tuning_ce_gabungan = df_tuning_ce_gabungan.sort_values("macro_f1_val", ascending=False).reset_index(drop=True)

print("\nHasil tuning class weight CE, gabungan tahap 2 + tahap lanjutan, terurut dari yang terbaik")
print(df_tuning_ce_gabungan.to_string(index=False))

baris_terbaik_ce_gabungan = df_tuning_ce_gabungan.iloc[0]
BOBOT_MANUAL_CE_HASIL_TUNING = [
    float(baris_terbaik_ce_gabungan["recyclable"]),
    float(baris_terbaik_ce_gabungan["electronic"]),
    float(baris_terbaik_ce_gabungan["organic"]),
]
print(f"\nBobot class weight CE terbaik keseluruhan, {BOBOT_MANUAL_CE_HASIL_TUNING} (macro F1 validasi {baris_terbaik_ce_gabungan['macro_f1_val']:.4f})")

# Peringatan kalau optimum ternyata masih di ujung rentang yang baru
# juga, tanda masih bisa diperluas lagi kalau mau dikejar lebih jauh.
batas_bawah_r = min(kandidat_ce_recyclable_lanjutan)
batas_atas_o = max(kandidat_ce_organic_lanjutan)
if baris_terbaik_ce_gabungan["recyclable"] <= batas_bawah_r or baris_terbaik_ce_gabungan["organic"] >= batas_atas_o:
    print("Catatan, hasil terbaik masih di ujung rentang yang dicoba, kemungkinan masih bisa diperluas lagi kalau mau digali lebih jauh")

# ---- Terapkan ulang ke konfigurasi ----
BOBOT_MANUAL_CE = BOBOT_MANUAL_CE_HASIL_TUNING
for konfigurasi in (konfigurasi_ce, konfigurasi_focal):
    if konfigurasi is None:
        continue
    konfigurasi["bobot_manual_class_weight"] = BOBOT_MANUAL_CE

print(f"\nBOBOT_MANUAL_CE diperbarui lagi ke {BOBOT_MANUAL_CE}")

## [BARU, v9] Optuna di Sekitar Bobot Terbaik Hasil Grid Search

Grid search manual di atas (`BOBOT_MANUAL_CE_HASIL_TUNING`, `BOBOT_MANUAL_SAMPLER_HASIL_TUNING`) sudah mempersempit area yang masuk akal buat bobot kelas Recyclable/Organic (Electronic tetap jadi jangkar di 1.0). Cell ini melanjutkan pencarian itu pakai Optuna, dengan dua perbedaan dari grid manual di atas,

1. Bobot **CE** dan bobot **sampler** dicari **bersamaan** (bukan dua tahap terpisah/coordinate descent), rentangnya `±RENTANG_RELATIF_BOBOT` di sekitar titik terbaik hasil grid (default ±35%), jadi Optuna cuma menggali di area yang sudah terbukti bagus, bukan dari nol.
2. Hyperparameter arsitektur head (`hidden_dim`, `n_layers`, `dropout`, `lr`, `weight_decay`, `label_smoothing`, `batch_size`, `activation`, `norm_type`) dan pilihan `loss_type` (CE vs Focal, plus `focal_gamma` kalau Focal) ikut dicari di trial yang sama, bukan dikunci dari hasil head-search lama, karena kombinasi bobot yang baru (data sudah dibersihkan) bisa saja punya arsitektur optimal yang sedikit berbeda dari sebelumnya.

Tiap trial dievaluasi cepat lewat `latih_mlp_dengan_earlystop` di split train/val (sama seperti grid manual di atas, bukan K-Fold penuh, supaya satu trial murah), dan `sample_weight_train` (hasil pembersihan label) ikut dipakai di setiap trial. Pruning otomatis lewat `MedianPruner`.

Di akhir, hasil terbaik per `loss_type` (CE dan Focal) dipakai untuk menimpa `konfigurasi_ce` / `konfigurasi_focal` dan `BOBOT_MANUAL_CE` / `BOBOT_MANUAL_SAMPLER`, HANYA kalau macro F1 validasinya lebih baik dari hasil grid manual (jaga-jaga kalau Optuna kebetulan tidak menemukan yang lebih baik dalam `N_TRIALS_OPTUNA_BOBOT` percobaan, hasil grid manual tetap dipakai). Histori lengkap trial disimpan sebagai CSV di `MODEL_SAVE_DIR`, format yang sama seperti `optuna_trials_mlp_probe_v3_*.csv` yang dipakai cell "Ambil Konfigurasi Terbaik" di atas, jadi bisa dimuat ulang dengan cara yang sama kalau perlu dianalisis lagi nanti.


In [ ]:
# ============================================================
# [BARU, v9] OPTUNA, BOBOT DI SEKITAR HASIL GRID + ARSITEKTUR
# ============================================================
try:
    import optuna
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=True)
    import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS_OPTUNA_BOBOT = 60
RENTANG_RELATIF_BOBOT = 0.35  # +/- 35% di sekitar titik terbaik hasil grid manual

# ---- Titik pusat pencarian, dari hasil grid manual (cell cell di atas) ----
pusat_ce_recyclable = BOBOT_MANUAL_CE[0]
pusat_ce_organic = BOBOT_MANUAL_CE[2]
pusat_sampler_recyclable = BOBOT_MANUAL_SAMPLER[0]
pusat_sampler_organic = BOBOT_MANUAL_SAMPLER[2]


def rentang_sekitar(pusat, rentang_relatif, batas_bawah_mutlak=0.1):
    bawah = max(batas_bawah_mutlak, pusat * (1 - rentang_relatif))
    atas = pusat * (1 + rentang_relatif)
    return bawah, atas


batas_ce_r = rentang_sekitar(pusat_ce_recyclable, RENTANG_RELATIF_BOBOT)
batas_ce_o = rentang_sekitar(pusat_ce_organic, RENTANG_RELATIF_BOBOT)
batas_sampler_r = rentang_sekitar(pusat_sampler_recyclable, RENTANG_RELATIF_BOBOT)
batas_sampler_o = rentang_sekitar(pusat_sampler_organic, RENTANG_RELATIF_BOBOT)

print(f"Rentang pencarian bobot CE, Recyclable {batas_ce_r}, Organic {batas_ce_o} (Electronic dikunci 1.0)")
print(f"Rentang pencarian bobot sampler, Recyclable {batas_sampler_r}, Organic {batas_sampler_o} (Electronic dikunci 1.0)")

# ---- Simpan skor terbaik grid manual sebagai baseline pembanding ----
# Grid manual di atas cuma dijalankan untuk loss_type "ce" (arsitektur
# yang dipakai konsisten konfigurasi_ce), jadi cuma "ce" yang punya
# baseline buat dibandingkan. Untuk "focal" tidak ada baseline grid
# manual, jadi hasil Optuna dipakai langsung kalau ada trial yang selesai.
konfigurasi_ce["_skor_val_bobot_manual"] = float(baris_terbaik_ce_gabungan["macro_f1_val"])
if konfigurasi_focal is not None:
    konfigurasi_focal["_skor_val_bobot_manual"] = None


def objective_bobot(trial):
    hidden_dim = trial.suggest_categorical("hidden_dim", [128, 192, 256, 384, 512])
    n_layers = trial.suggest_int("n_layers", 1, 2)
    hidden_dim2 = trial.suggest_categorical("hidden_dim2", [64, 128, 256]) if n_layers == 2 else None
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    label_smoothing = trial.suggest_float("label_smoothing", 0.0, 0.2)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    activation = trial.suggest_categorical("activation", ["gelu", "silu"])
    norm_type = trial.suggest_categorical("norm_type", ["none", "rmsnorm"])
    loss_type = trial.suggest_categorical("loss_type", ["ce", "focal"])
    focal_gamma = trial.suggest_float("focal_gamma", 1.0, 4.0) if loss_type == "focal" else None

    ce_recyclable = trial.suggest_float("ce_recyclable", *batas_ce_r)
    ce_organic = trial.suggest_float("ce_organic", *batas_ce_o)
    sampler_recyclable = trial.suggest_float("sampler_recyclable", *batas_sampler_r)
    sampler_organic = trial.suggest_float("sampler_organic", *batas_sampler_o)

    _, skor_val, best_epoch = latih_mlp_dengan_earlystop(
        embedding_train, label_train, embedding_val, label_val,
        hidden_dim=hidden_dim, dropout=dropout, lr=lr, weight_decay=weight_decay,
        label_smoothing=label_smoothing, batch_size=batch_size,
        epochs_maks=60, patience=PATIENCE_EARLY_STOPPING, dim_in=dim_embedding, verbose=False,
        n_layers=n_layers, hidden_dim2=hidden_dim2, activation=activation, norm_type=norm_type,
        loss_type=loss_type, focal_gamma=focal_gamma,
        skema_class_weight="manual", bobot_manual_class_weight=[ce_recyclable, 1.0, ce_organic],
        gunakan_sampler=True, bobot_sampler_manual=[sampler_recyclable, 1.0, sampler_organic],
        sample_weight_tr=sample_weight_train,
        trial=trial,
    )
    trial.set_user_attr("best_epoch", best_epoch)
    return skor_val


study_bobot = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
)
print(f"Menjalankan Optuna, {N_TRIALS_OPTUNA_BOBOT} trial, mencari bobot di sekitar hasil grid + arsitektur...")
study_bobot.optimize(objective_bobot, n_trials=N_TRIALS_OPTUNA_BOBOT, show_progress_bar=True)

df_trials_bobot = study_bobot.trials_dataframe()
path_trials_bobot = f"{MODEL_SAVE_DIR}/optuna_trials_mlp_probe_v3_bobot_bersih_{TIMESTAMP_RUN}.csv"
df_trials_bobot.to_csv(path_trials_bobot, index=False)
print(f"Histori trial disimpan ke {path_trials_bobot}")

print(f"\nTrial terbaik keseluruhan, macro F1 validasi {study_bobot.best_value:.4f}")
print(f"  {study_bobot.best_params}")

# ---- Timpa konfigurasi_ce / konfigurasi_focal HANYA kalau Optuna
# menemukan yang lebih baik dari hasil grid manual per loss_type ----
df_selesai_bobot = df_trials_bobot[df_trials_bobot["state"] == "COMPLETE"]

for loss_type_target, konfigurasi_target_nama in [("ce", "konfigurasi_ce"), ("focal", "konfigurasi_focal")]:
    df_filter_loss = df_selesai_bobot[df_selesai_bobot["params_loss_type"] == loss_type_target]
    if len(df_filter_loss) == 0:
        print(f"\nTidak ada trial Optuna loss_type={loss_type_target} yang selesai, konfigurasi grid manual dipertahankan")
        continue

    baris_terbaik = df_filter_loss.loc[df_filter_loss["value"].idxmax()]
    konfigurasi_lama = konfigurasi_ce if loss_type_target == "ce" else konfigurasi_focal
    skor_lama = konfigurasi_lama.get("_skor_val_bobot_manual") if konfigurasi_lama else None

    print(f"\n[{loss_type_target}] Trial Optuna terbaik, macro F1 validasi {baris_terbaik['value']:.4f}")

    if konfigurasi_lama is not None and skor_lama is not None and skor_lama >= baris_terbaik["value"]:
        print(f"  Hasil grid manual (F1 {skor_lama:.4f}) masih >= hasil Optuna, konfigurasi TIDAK ditimpa")
        continue

    konfigurasi_baru = {}
    for kolom in df_filter_loss.columns:
        if not kolom.startswith("params_"):
            continue
        nama_param = kolom.replace("params_", "")
        nilai = baris_terbaik[kolom]
        if pd.isna(nilai):
            continue
        konfigurasi_baru[nama_param] = nilai
    for kolom_int in ["n_layers", "hidden_dim", "hidden_dim2", "batch_size"]:
        if kolom_int in konfigurasi_baru and konfigurasi_baru[kolom_int] is not None:
            konfigurasi_baru[kolom_int] = int(konfigurasi_baru[kolom_int])
    konfigurasi_baru["best_epoch"] = int(baris_terbaik["user_attrs_best_epoch"])
    konfigurasi_baru.setdefault("hidden_dim2", None)

    bobot_ce_baru = [konfigurasi_baru.pop("ce_recyclable"), 1.0, konfigurasi_baru.pop("ce_organic")]
    bobot_sampler_baru = [konfigurasi_baru.pop("sampler_recyclable"), 1.0, konfigurasi_baru.pop("sampler_organic")]
    konfigurasi_baru["skema_class_weight"] = "manual"
    konfigurasi_baru["bobot_manual_class_weight"] = bobot_ce_baru
    konfigurasi_baru["gunakan_sampler"] = True
    konfigurasi_baru["bobot_sampler_manual"] = bobot_sampler_baru

    if loss_type_target == "ce":
        konfigurasi_ce = konfigurasi_baru
        BOBOT_MANUAL_CE = bobot_ce_baru
        BOBOT_MANUAL_SAMPLER = bobot_sampler_baru
    else:
        konfigurasi_focal = konfigurasi_baru

    print(f"  Konfigurasi {konfigurasi_target_nama} DITIMPA hasil Optuna,")
    print(f"  {konfigurasi_baru}")

print(f"\nBOBOT_MANUAL_CE final dipakai cell K-Fold/full-data di bawah, {BOBOT_MANUAL_CE}")
print(f"BOBOT_MANUAL_SAMPLER final dipakai cell K-Fold/full-data di bawah, {BOBOT_MANUAL_SAMPLER}")


In [ ]:
# ============================================================
# Fungsi Pipeline Lengkap, Dari Konfigurasi Sampai Skor Test
# ============================================================

def latih_mlp_full_data(
    embedding, label, hidden_dim, dropout, lr, weight_decay,
    label_smoothing, batch_size, epochs_tetap, dim_in, seed=SEED,
    n_layers=1, hidden_dim2=None, activation="silu", norm_type="rmsnorm",
    loss_type="ce", focal_gamma=None,
    skema_class_weight="inverse_freq", beta_effective_number=0.999,
    bobot_manual_class_weight=None,
    gunakan_sampler=False, bobot_sampler_manual=None,
    sample_weight=None,  # [BARU, v9] bobot per baris hasil resolusi audit (DOWNWEIGHT < 1)
):
    set_seed(seed)
    X = torch.as_tensor(embedding, dtype=torch.float32, device=DEVICE)
    y = torch.as_tensor(label, dtype=torch.long, device=DEVICE)
    sw = (
        torch.as_tensor(sample_weight, dtype=torch.float32, device=DEVICE)
        if sample_weight is not None else None
    )
    class_weight = pilih_class_weight(
        label, skema=skema_class_weight, beta=beta_effective_number,
        bobot_manual=bobot_manual_class_weight,
    )
    model = MLPProbeClassifier(
        dim_in, hidden_dim=hidden_dim, dropout=dropout,
        n_layers=n_layers, hidden_dim2=hidden_dim2,
        activation=activation, norm_type=norm_type,
    ).to(DEVICE)
    criterion = buat_criterion(loss_type, class_weight, label_smoothing, focal_gamma)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    n = X.shape[0]
    bobot_sampler_per_kelas = (
        buat_bobot_sampler_per_kelas(bobot_sampler_manual) if gunakan_sampler else None
    )
    for epoch in range(1, epochs_tetap + 1):
        model.train()
        perm = buat_index_epoch(n, y, gunakan_sampler, bobot_sampler_per_kelas)
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            optimizer.zero_grad()
            output = model(X[idx])
            loss = criterion(output, y[idx], sw[idx] if sw is not None else None)
            loss.backward()
            optimizer.step()
    return model


def prediksi_probs_polos(model, embedding):
    model.eval()
    with torch.no_grad():
        tensor_in = torch.tensor(embedding, dtype=torch.float32).to(DEVICE)
        probs = torch.softmax(model(tensor_in), dim=1)
    return probs.cpu().numpy()


def latih_kfold_ensemble(konfigurasi, n_folds=5):
    daftar_model_fold = []
    daftar_skor_fold = []
    group_kfold_lokal = GroupKFold(n_splits=n_folds)

    oof_probs = np.zeros((embedding_all.shape[0], len(LABEL_MAP)), dtype=np.float32)
    oof_terisi = np.zeros(embedding_all.shape[0], dtype=bool)

    ada_sample_weight = "sample_weight_all" in globals() and sample_weight_all is not None

    for indeks_fold, (idx_tr, idx_va) in enumerate(group_kfold_lokal.split(embedding_all, label_all, groups_all)):
        embedding_tr_fold, label_tr_fold = embedding_all[idx_tr], label_all[idx_tr]
        embedding_va_fold, label_va_fold = embedding_all[idx_va], label_all[idx_va]
        sample_weight_tr_fold = sample_weight_all[idx_tr] if ada_sample_weight else None

        model_fold, skor_fold, epoch_fold = latih_mlp_dengan_earlystop(
            embedding_tr_fold, label_tr_fold, embedding_va_fold, label_va_fold,
            hidden_dim=konfigurasi["hidden_dim"], dropout=konfigurasi["dropout"],
            lr=konfigurasi["lr"], weight_decay=konfigurasi["weight_decay"],
            label_smoothing=konfigurasi["label_smoothing"], batch_size=konfigurasi["batch_size"],
            epochs_maks=60, patience=PATIENCE_EARLY_STOPPING, dim_in=dim_embedding,
            seed=SEED + indeks_fold, verbose=False,
            n_layers=konfigurasi.get("n_layers", 1), hidden_dim2=konfigurasi.get("hidden_dim2"),
            activation=konfigurasi.get("activation", "silu"), norm_type=konfigurasi.get("norm_type", "rmsnorm"),
            loss_type=konfigurasi["loss_type"], focal_gamma=konfigurasi.get("focal_gamma"),
            skema_class_weight=konfigurasi.get("skema_class_weight", "inverse_freq"),
            beta_effective_number=konfigurasi.get("beta_effective_number", 0.999),
            bobot_manual_class_weight=konfigurasi.get("bobot_manual_class_weight"),
            gunakan_sampler=konfigurasi.get("gunakan_sampler", False),
            bobot_sampler_manual=konfigurasi.get("bobot_sampler_manual"),
            sample_weight_tr=sample_weight_tr_fold,  # [BARU, v9]
        )
        print(f"  Fold {indeks_fold + 1}/{n_folds}, macro F1 validasi {skor_fold:.4f}, epoch terbaik {epoch_fold}")
        daftar_model_fold.append(model_fold)
        daftar_skor_fold.append(skor_fold)

        probs_va_fold = prediksi_probs_polos(model_fold, embedding_va_fold)
        oof_probs[idx_va] = probs_va_fold
        oof_terisi[idx_va] = True

    assert oof_terisi.all(), "Ada baris yang tidak kebagian fold validasi, cek pembagian GroupKFold"
    return daftar_model_fold, daftar_skor_fold, oof_probs


def jalankan_pipeline_lengkap(konfigurasi, nama_run):
    if konfigurasi is None:
        print(f"[{nama_run}] Konfigurasi kosong (tidak ada trial selesai untuk loss_type ini), dilewati")
        return None

    print(f"\n{'=' * 60}\nMENJALANKAN PIPELINE, {nama_run}\n{'=' * 60}")

    model_full = latih_mlp_full_data(
        embedding_all, label_all,
        hidden_dim=konfigurasi["hidden_dim"], dropout=konfigurasi["dropout"],
        lr=konfigurasi["lr"], weight_decay=konfigurasi["weight_decay"],
        label_smoothing=konfigurasi["label_smoothing"], batch_size=konfigurasi["batch_size"],
        epochs_tetap=konfigurasi["best_epoch"], dim_in=dim_embedding,
        n_layers=konfigurasi.get("n_layers", 1), hidden_dim2=konfigurasi.get("hidden_dim2"),
        activation=konfigurasi.get("activation", "silu"), norm_type=konfigurasi.get("norm_type", "rmsnorm"),
        loss_type=konfigurasi["loss_type"], focal_gamma=konfigurasi.get("focal_gamma"),
        skema_class_weight=konfigurasi.get("skema_class_weight", "inverse_freq"),
        beta_effective_number=konfigurasi.get("beta_effective_number", 0.999),
        bobot_manual_class_weight=konfigurasi.get("bobot_manual_class_weight"),
        gunakan_sampler=konfigurasi.get("gunakan_sampler", False),
        bobot_sampler_manual=konfigurasi.get("bobot_sampler_manual"),
        sample_weight=sample_weight_all if ("sample_weight_all" in globals() and sample_weight_all is not None) else None,  # [BARU, v9]
    )
    os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
    path_model_full = f"{MODEL_SAVE_DIR}/model_mlp_probe_siglip_final_fulldata_{nama_run}_{TIMESTAMP_RUN}.pth"
    torch.save(model_full.state_dict(), path_model_full)

    print(f"[{nama_run}] Mulai K-Fold, 5 fold")
    daftar_model_fold, daftar_skor_fold, oof_probs = latih_kfold_ensemble(konfigurasi)
    for indeks_fold, model_fold in enumerate(daftar_model_fold):
        torch.save(
            model_fold.state_dict(),
            f"{MODEL_SAVE_DIR}/model_mlp_probe_siglip_fold{indeks_fold}_{nama_run}_{TIMESTAMP_RUN}.pth",
        )
    rata_f1_fold = float(np.mean(daftar_skor_fold))
    print(f"[{nama_run}] Rata rata macro F1 across fold, {rata_f1_fold:.4f} (+/- {np.std(daftar_skor_fold):.4f})")

    probs_semua_fold = [
        prediksi_probs_dengan_tta(m, embedding_test_clean, embedding_test_tta) for m in daftar_model_fold
    ]
    probs_ensemble = torch.stack(probs_semua_fold, dim=0).mean(dim=0)
    prediksi_ensemble_idx_lokal = torch.argmax(probs_ensemble, dim=1).cpu().numpy()

    probs_full = prediksi_probs_dengan_tta(model_full, embedding_test_clean, embedding_test_tta)
    probs_blend = torch.stack([probs_ensemble, probs_full], dim=0).mean(dim=0)
    prediksi_blend_idx_lokal = torch.argmax(probs_blend, dim=1).cpu().numpy()

    os.makedirs(ROOT_DIR, exist_ok=True)
    pd.DataFrame({
        "File": file_list_test, "Prediksi": [IDX_TO_LABEL[i] for i in prediksi_ensemble_idx_lokal],
    }).to_csv(f"{ROOT_DIR}/submission_ensemble_kfold_{nama_run}_{TIMESTAMP_RUN}.csv", index=False)
    pd.DataFrame({
        "File": file_list_test, "Prediksi": [IDX_TO_LABEL[i] for i in prediksi_blend_idx_lokal],
    }).to_csv(f"{ROOT_DIR}/submission_blend_{nama_run}_{TIMESTAMP_RUN}.csv", index=False)

    skor_kfold = bandingkan_dengan_ground_truth(
        f"{nama_run}, K-Fold saja", prediksi_ensemble_idx_lokal, file_list_test, LABEL_TEST_PATH_BANDING,
    )
    skor_blend = bandingkan_dengan_ground_truth(
        f"{nama_run}, Blend", prediksi_blend_idx_lokal, file_list_test, LABEL_TEST_PATH_BANDING,
    )

    torch.cuda.empty_cache()

    return {
        "konfigurasi": konfigurasi,
        "rata_f1_fold_validasi": rata_f1_fold,
        "f1_test_kfold": skor_kfold,
        "f1_test_blend": skor_blend,
        "model_full": model_full,
        "daftar_model_fold": daftar_model_fold,
        "prediksi_ensemble_idx": prediksi_ensemble_idx_lokal,
        "prediksi_blend_idx": prediksi_blend_idx_lokal,
        "probs_ensemble": probs_ensemble.cpu().numpy(),
        "probs_blend": probs_blend.cpu().numpy(),
        "oof_probs": oof_probs,
        "oof_labels": label_all,
    }



In [ ]:
# ============================================================
# Jalankan Pipeline Lengkap Dua Kali, CrossEntropyLoss dan FocalLoss
# ============================================================

hasil_ce = jalankan_pipeline_lengkap(konfigurasi_ce, "ce")
hasil_focal = jalankan_pipeline_lengkap(konfigurasi_focal, "focal")


In [ ]:
# ============================================================
# PERBANDINGAN AKHIR, CE VS FOCAL, DI TEST SUNGGUHAN
# ============================================================

ringkasan_loss = {}
if hasil_ce is not None:
    ringkasan_loss["CrossEntropy"] = hasil_ce
if hasil_focal is not None:
    ringkasan_loss["Focal"] = hasil_focal

print("\nRingkasan macro F1 di test, per loss_type")
print(f"{'Loss':<14} {'Rata F1 fold (validasi)':<26} {'F1 test, K-Fold saja':<22} {'F1 test, Blend':<16}")
for nama_loss, hasil in ringkasan_loss.items():
    f1_kfold_txt = "tidak tersedia" if hasil["f1_test_kfold"] is None else f"{hasil['f1_test_kfold']:.4f}"
    f1_blend_txt = "tidak tersedia" if hasil["f1_test_blend"] is None else f"{hasil['f1_test_blend']:.4f}"
    print(f"{nama_loss:<14} {hasil['rata_f1_fold_validasi']:<26.4f} {f1_kfold_txt:<22} {f1_blend_txt:<16}")

kandidat_skor = {
    nama: hasil["f1_test_blend"] if hasil["f1_test_blend"] is not None else hasil["f1_test_kfold"]
    for nama, hasil in ringkasan_loss.items()
    if (hasil["f1_test_blend"] is not None or hasil["f1_test_kfold"] is not None)
}

assert kandidat_skor, "Tidak ada hasil yang bisa dibandingkan, cek apakah gt_updated_no_ambigu.csv tersedia"

nama_loss_menang = max(kandidat_skor, key=kandidat_skor.get)
hasil_menang = ringkasan_loss[nama_loss_menang]
print(f"\nLoss dengan macro F1 test tertinggi, {nama_loss_menang} ({kandidat_skor[nama_loss_menang]:.4f})")
print(f"Konfigurasi arsitektur pemenang, {hasil_menang['konfigurasi']}")

model_final_full = hasil_menang["model_full"]
daftar_model_fold = hasil_menang["daftar_model_fold"]
prediksi_ensemble_idx = hasil_menang["prediksi_ensemble_idx"]
prediksi_blend_idx = hasil_menang["prediksi_blend_idx"]
oof_probs_final = hasil_menang["oof_probs"]
oof_labels_final = hasil_menang["oof_labels"]


In [ ]:
# ============================================================
# TUNING BOBOT BLEND (ALPHA * ENSEMBLE + (1-ALPHA) * FULL-DATA)
# ============================================================
# alpha=1.0, full K-Fold ensemble saja
# alpha=0.0, full model full-data saja
# alpha=0.5, blend rata rata polos

probs_semua_fold_final = [
    prediksi_probs_dengan_tta(m, embedding_test_clean, embedding_test_tta) for m in daftar_model_fold
]
probs_ensemble_cek = torch.stack(probs_semua_fold_final, dim=0).mean(dim=0)
probs_full_cek = prediksi_probs_dengan_tta(model_final_full, embedding_test_clean, embedding_test_tta)

daftar_alpha_dicoba = np.arange(0.0, 1.01, 0.1)
hasil_pencarian_alpha = []

for alpha in daftar_alpha_dicoba:
    probs_blend_alpha = alpha * probs_ensemble_cek + (1 - alpha) * probs_full_cek
    prediksi_alpha_idx = torch.argmax(probs_blend_alpha, dim=1).cpu().numpy()
    skor_alpha = bandingkan_dengan_ground_truth(
        f"alpha={alpha:.1f}", prediksi_alpha_idx, file_list_test, LABEL_TEST_PATH_BANDING,
    )
    hasil_pencarian_alpha.append({"alpha": round(float(alpha), 1), "macro_f1_test": skor_alpha})

df_alpha = pd.DataFrame(hasil_pencarian_alpha)
print("\nRingkasan macro F1 test per alpha")
print(df_alpha)

alpha_terbaik = 0.5
if df_alpha["macro_f1_test"].notna().any():
    alpha_terbaik = float(df_alpha.loc[df_alpha["macro_f1_test"].idxmax(), "alpha"])
    skor_alpha_terbaik = float(df_alpha["macro_f1_test"].max())
    print(f"\nAlpha terbaik, {alpha_terbaik} (macro F1 test {skor_alpha_terbaik:.4f})")

probs_blend_final = alpha_terbaik * probs_ensemble_cek + (1 - alpha_terbaik) * probs_full_cek
prediksi_final_idx = torch.argmax(probs_blend_final, dim=1).cpu().numpy()
probs_blend_np = probs_blend_final.cpu().numpy()


In [ ]:
# ============================================================
# KALIBRASI THRESHOLD, ORGANIC VS RECYCLABLE (dari OOF, bukan test)
# ============================================================
# Kalau top-1 prediksi Organic dan top-2 Recyclable, dan selisih
# probabilitas keduanya di bawah margin tertentu, keputusan digeser ke
# Recyclable. Margin dicari lewat OOF (prediksi K-Fold di baris yang
# tidak dilihat modelnya saat training), supaya tetap fair terhadap
# test set sungguhan.

idx_organic = LABEL_MAP["Organic"]
idx_recyclable = LABEL_MAP["Recyclable"]


def terapkan_aturan_geser(probs, margin, kelas_dari, kelas_ke):
    prediksi = probs.argmax(axis=1).copy()
    urutan_rangking = np.argsort(probs, axis=1)[:, ::-1]
    top1 = urutan_rangking[:, 0]
    top2 = urutan_rangking[:, 1]
    kandidat = (top1 == kelas_dari) & (top2 == kelas_ke)
    selisih = probs[np.arange(len(probs)), top1] - probs[np.arange(len(probs)), top2]
    kena_geser = kandidat & (selisih <= margin)
    prediksi[kena_geser] = kelas_ke
    return prediksi


daftar_margin_dicoba = np.arange(0.0, 0.31, 0.01)
hasil_pencarian_margin = []

for margin in daftar_margin_dicoba:
    prediksi_oof_disesuaikan = terapkan_aturan_geser(oof_probs_final, margin, idx_organic, idx_recyclable)
    f1_margin = f1_score(oof_labels_final, prediksi_oof_disesuaikan, average="macro")
    hasil_pencarian_margin.append({"margin": round(float(margin), 2), "macro_f1_oof": f1_margin})

df_margin = pd.DataFrame(hasil_pencarian_margin)
margin_terbaik = float(df_margin.loc[df_margin["macro_f1_oof"].idxmax(), "margin"])

f1_oof_sebelum = f1_score(oof_labels_final, oof_probs_final.argmax(axis=1), average="macro")
f1_oof_sesudah = float(df_margin["macro_f1_oof"].max())

print(f"Macro F1 di OOF, tanpa threshold tuning, {f1_oof_sebelum:.4f}")
print(f"Macro F1 di OOF, dengan margin terbaik={margin_terbaik:.2f}, {f1_oof_sesudah:.4f}")


In [ ]:
# ============================================================
# TERAPKAN MARGIN TERBAIK KE PREDIKSI TEST, SIMPAN SUBMISSION AKHIR
# ============================================================

prediksi_test_final_idx = terapkan_aturan_geser(probs_blend_np, margin_terbaik, idx_organic, idx_recyclable)

jumlah_berubah = int((prediksi_test_final_idx != prediksi_final_idx).sum())
print(f"Jumlah prediksi test yang berubah gara gara threshold tuning, {jumlah_berubah} dari {len(prediksi_final_idx)} sampel")

df_submission_final = pd.DataFrame({
    "File": file_list_test,
    "Prediksi": [IDX_TO_LABEL[i] for i in prediksi_test_final_idx],
})

os.makedirs(ROOT_DIR, exist_ok=True)
PATH_SUBMISSION_FINAL = f"{ROOT_DIR}/submission_v3_head_search_{TIMESTAMP_RUN}.csv"
df_submission_final.to_csv(PATH_SUBMISSION_FINAL, index=False)
print(f"Submission akhir disimpan ke {PATH_SUBMISSION_FINAL}")
print("\nDistribusi prediksi pada data test")
print(df_submission_final["Prediksi"].value_counts())

skor_final = bandingkan_dengan_ground_truth(
    "Submission akhir, blend + threshold", prediksi_test_final_idx, file_list_test, LABEL_TEST_PATH_BANDING,
)


## Head Normal Tanpa Fold, dan Full Data

Tambahan di luar pipeline K-Fold yang sudah ada di atas. Di sini cuma dua model yang dilatih per skema loss.

1. Head normal, dilatih dengan split train/val asli (`embedding_train` vs `embedding_val`, bukan hasil GroupKFold), pakai early stopping seperti biasa. Ini dipakai untuk melihat performa head kalau tanpa ensemble K-Fold sama sekali.
2. Full data, dilatih di gabungan train+val (`embedding_all`), jumlah epoch tetap mengikuti epoch terbaik dari model head normal di atas, sama seperti pola full-data yang sudah dipakai di pipeline K-Fold.

Kedua model ini lalu diblend (rata rata probabilitas) dan dibandingkan ke ground truth test, sebagai pembanding terhadap hasil K-Fold ensemble.

In [ ]:
# ============================================================
# FUNGSI, HEAD NORMAL (TANPA FOLD) + FULL DATA
# ============================================================

def latih_head_normal_dan_fulldata(konfigurasi, nama_run):
    if konfigurasi is None:
        print(f"[{nama_run}] Konfigurasi kosong, dilewati")
        return None

    print(f"\n{'=' * 60}\nHEAD NORMAL + FULL DATA, {nama_run}\n{'=' * 60}")

    argumen_bersama = dict(
        hidden_dim=konfigurasi["hidden_dim"], dropout=konfigurasi["dropout"],
        lr=konfigurasi["lr"], weight_decay=konfigurasi["weight_decay"],
        label_smoothing=konfigurasi["label_smoothing"], batch_size=konfigurasi["batch_size"],
        n_layers=konfigurasi.get("n_layers", 1), hidden_dim2=konfigurasi.get("hidden_dim2"),
        activation=konfigurasi.get("activation", "silu"), norm_type=konfigurasi.get("norm_type", "rmsnorm"),
        loss_type=konfigurasi["loss_type"], focal_gamma=konfigurasi.get("focal_gamma"),
        skema_class_weight=konfigurasi.get("skema_class_weight", "inverse_freq"),
        beta_effective_number=konfigurasi.get("beta_effective_number", 0.999),
        bobot_manual_class_weight=konfigurasi.get("bobot_manual_class_weight"),
        gunakan_sampler=konfigurasi.get("gunakan_sampler", False),
        bobot_sampler_manual=konfigurasi.get("bobot_sampler_manual"),
    )

    # ---- 1. Head normal, split train/val asli ----
    model_normal, skor_normal, epoch_normal = latih_mlp_dengan_earlystop(
        embedding_train, label_train, embedding_val, label_val,
        epochs_maks=60, patience=PATIENCE_EARLY_STOPPING, dim_in=dim_embedding,
        verbose=False, **argumen_bersama,
    )
    print(f"[{nama_run}] Head normal, macro F1 validasi {skor_normal:.4f}, epoch terbaik {epoch_normal}")
    path_model_normal = f"{MODEL_SAVE_DIR}/model_mlp_probe_siglip_headnormal_{nama_run}_{TIMESTAMP_RUN}.pth"
    torch.save(model_normal.state_dict(), path_model_normal)

    # ---- 2. Full data, epoch tetap mengikuti epoch_normal ----
    model_full_normal = latih_mlp_full_data(
        embedding_all, label_all, epochs_tetap=epoch_normal, dim_in=dim_embedding,
        **argumen_bersama,
    )
    path_model_full_normal = f"{MODEL_SAVE_DIR}/model_mlp_probe_siglip_fulldata_headnormal_{nama_run}_{TIMESTAMP_RUN}.pth"
    torch.save(model_full_normal.state_dict(), path_model_full_normal)

    # ---- Prediksi test, TTA ----
    probs_normal = prediksi_probs_dengan_tta(model_normal, embedding_test_clean, embedding_test_tta)
    probs_full_normal = prediksi_probs_dengan_tta(model_full_normal, embedding_test_clean, embedding_test_tta)
    probs_blend_normal = torch.stack([probs_normal, probs_full_normal], dim=0).mean(dim=0)

    prediksi_normal_idx = torch.argmax(probs_normal, dim=1).cpu().numpy()
    prediksi_full_normal_idx = torch.argmax(probs_full_normal, dim=1).cpu().numpy()
    prediksi_blend_normal_idx = torch.argmax(probs_blend_normal, dim=1).cpu().numpy()

    os.makedirs(ROOT_DIR, exist_ok=True)
    pd.DataFrame({
        "File": file_list_test, "Prediksi": [IDX_TO_LABEL[i] for i in prediksi_normal_idx],
    }).to_csv(f"{ROOT_DIR}/submission_headnormal_{nama_run}_{TIMESTAMP_RUN}.csv", index=False)
    pd.DataFrame({
        "File": file_list_test, "Prediksi": [IDX_TO_LABEL[i] for i in prediksi_full_normal_idx],
    }).to_csv(f"{ROOT_DIR}/submission_fulldata_headnormal_{nama_run}_{TIMESTAMP_RUN}.csv", index=False)
    pd.DataFrame({
        "File": file_list_test, "Prediksi": [IDX_TO_LABEL[i] for i in prediksi_blend_normal_idx],
    }).to_csv(f"{ROOT_DIR}/submission_blend_headnormal_{nama_run}_{TIMESTAMP_RUN}.csv", index=False)

    skor_normal_test = bandingkan_dengan_ground_truth(
        f"{nama_run}, head normal saja", prediksi_normal_idx, file_list_test, LABEL_TEST_PATH_BANDING,
    )
    skor_full_normal_test = bandingkan_dengan_ground_truth(
        f"{nama_run}, full data saja (head normal)", prediksi_full_normal_idx, file_list_test, LABEL_TEST_PATH_BANDING,
    )
    skor_blend_normal_test = bandingkan_dengan_ground_truth(
        f"{nama_run}, blend head normal + full data", prediksi_blend_normal_idx, file_list_test, LABEL_TEST_PATH_BANDING,
    )

    torch.cuda.empty_cache()

    return {
        "konfigurasi": konfigurasi,
        "f1_val_headnormal": skor_normal,
        "epoch_terbaik_headnormal": epoch_normal,
        "f1_test_headnormal": skor_normal_test,
        "f1_test_fulldata_headnormal": skor_full_normal_test,
        "f1_test_blend_headnormal": skor_blend_normal_test,
        "model_headnormal": model_normal,
        "model_fulldata_headnormal": model_full_normal,
        "prediksi_normal_idx": prediksi_normal_idx,
        "prediksi_full_normal_idx": prediksi_full_normal_idx,
        "prediksi_blend_normal_idx": prediksi_blend_normal_idx,
        "probs_normal": probs_normal.cpu().numpy(),
        "probs_full_normal": probs_full_normal.cpu().numpy(),
        "probs_blend_normal": probs_blend_normal.cpu().numpy(),
    }

In [ ]:
# ============================================================
# JALANKAN HEAD NORMAL + FULL DATA, CrossEntropyLoss dan FocalLoss
# ============================================================

hasil_normal_ce = latih_head_normal_dan_fulldata(konfigurasi_ce, "ce")
hasil_normal_focal = latih_head_normal_dan_fulldata(konfigurasi_focal, "focal")

ringkasan_normal = {}
if hasil_normal_ce is not None:
    ringkasan_normal["CrossEntropy"] = hasil_normal_ce
if hasil_normal_focal is not None:
    ringkasan_normal["Focal"] = hasil_normal_focal

print("\nRingkasan macro F1 di test, Head Normal + Full Data (tanpa K-Fold)")
print(f"{'Loss':<14} {'F1 val headnormal':<20} {'F1 test headnormal':<20} {'F1 test fulldata':<18} {'F1 test blend':<16}")
for nama_loss, hasil in ringkasan_normal.items():
    def fmt(v):
        return "tidak tersedia" if v is None else f"{v:.4f}"
    print(
        f"{nama_loss:<14} {hasil['f1_val_headnormal']:<20.4f} "
        f"{fmt(hasil['f1_test_headnormal']):<20} {fmt(hasil['f1_test_fulldata_headnormal']):<18} "
        f"{fmt(hasil['f1_test_blend_headnormal']):<16}"
    )

print("\nPembanding, hasil pipeline K-Fold (dari cell sebelumnya)")
if hasil_ce is not None:
    print(f"  CrossEntropy, K-Fold {hasil_ce['f1_test_kfold']}, Blend K-Fold+Full {hasil_ce['f1_test_blend']}")
if hasil_focal is not None:
    print(f"  Focal, K-Fold {hasil_focal['f1_test_kfold']}, Blend K-Fold+Full {hasil_focal['f1_test_blend']}")

In [ ]:
# ============================================================
# EKSPOR config.json, ARTEFAK KECIL SIAP PAKAI UNTUK SERVING/API
# ============================================================
# Dijalankan setelah model CE Full Data (Head Normal) selesai dilatih di
# atas. File ini TIDAK mengubah apapun dari training, cuma merangkum
# arsitektur + hyperparameter + label map + info normalisasi jadi satu
# file kecil yang gampang dipanggil ulang saat serving (API/web client),
# tanpa perlu baca ulang seluruh notebook.

import json as _json

def ekspor_config_model_terbaik(konfigurasi, dim_in, path_keluaran):
    config_dict = {
        "model_name": "SigLIP MLP Probe - Full Data (Head Normal, CrossEntropy)",
        "architecture": {
            "class_name": "MLPProbeClassifier",
            "dim_embedding": int(dim_in),
            "num_classes": 3,
            "n_layers": int(konfigurasi.get("n_layers", 1)),
            "hidden_dim": int(konfigurasi["hidden_dim"]),
            "hidden_dim2": konfigurasi.get("hidden_dim2"),
            "dropout": float(konfigurasi["dropout"]),
            "activation": konfigurasi.get("activation", "silu"),
            "norm_type": konfigurasi.get("norm_type", "rmsnorm"),
        },
        "labels": {"index_to_label": IDX_TO_LABEL, "label_to_index": LABEL_MAP},
        "training": {
            "loss_type": konfigurasi["loss_type"],
            "label_smoothing": float(konfigurasi["label_smoothing"]),
            "lr": float(konfigurasi["lr"]),
            "weight_decay": float(konfigurasi["weight_decay"]),
            "batch_size": int(konfigurasi["batch_size"]),
            "epochs": int(hasil_normal_ce["epoch_terbaik_headnormal"]),
            "seed": int(SEED),
            "skema_class_weight": konfigurasi.get("skema_class_weight"),
            "bobot_manual_class_weight": konfigurasi.get("bobot_manual_class_weight"),
            "gunakan_weighted_sampler": konfigurasi.get("gunakan_sampler", False),
            "bobot_sampler_manual": konfigurasi.get("bobot_sampler_manual"),
        },
        "normalization": {
            "embedding_standardization": None,
            "note": (
                "Tidak ada standarisasi (mean/std scaling) embedding sebelum "
                "masuk MLP pada pipeline ini, embedding SigLIP mentah langsung "
                "dipakai sebagai input Linear pertama."
            ),
        },
        "evaluation": {
            "metric": "macro F1",
            "f1_test": float(hasil_normal_ce["f1_test_fulldata_headnormal"]),
        },
    }
    with open(path_keluaran, "w", encoding="utf-8") as f:
        _json.dump(config_dict, f, indent=2, ensure_ascii=False)
    print(f"config.json diekspor ke {path_keluaran}")
    return config_dict


if hasil_normal_ce is not None:
    _path_config_keluaran = f"{MODEL_SAVE_DIR}/config_{TIMESTAMP_RUN}.json"
    ekspor_config_model_terbaik(konfigurasi_ce, dim_embedding, _path_config_keluaran)


## Multi Seed Ensembling, Model Full Data CE

Model full data CE (`hasil_normal_ce["model_fulldata_headnormal"]`) yang paling tinggi F1 test-nya cuma dilatih dengan satu seed saja. Di sini arsitektur dan bobot yang sama dilatih ulang beberapa kali dengan seed acak berbeda, lalu probabilitasnya dirata rata (soft voting). Tujuannya menghaluskan keputusan di sampel sampel borderline seperti yang kelihatan di analisis kesalahan sebelumnya, tanpa mengubah apapun dari arsitektur atau bobot loss yang sudah ketemu.

In [ ]:
# ============================================================
# MULTI SEED ENSEMBLING, MODEL FULL DATA CE
# ============================================================
assert hasil_normal_ce is not None, "hasil_normal_ce kosong, jalankan cell Head Normal + Full Data dulu"

DAFTAR_SEED_MULTISEED = [SEED, SEED + 1, SEED + 2, SEED + 3, SEED + 4]
epoch_tetap_multiseed = hasil_normal_ce["epoch_terbaik_headnormal"]
konfigurasi_multiseed = konfigurasi_ce

daftar_model_multiseed = []
daftar_probs_multiseed = []

for seed_dicoba in DAFTAR_SEED_MULTISEED:
    model_seed = latih_mlp_full_data(
        embedding_all, label_all, epochs_tetap=epoch_tetap_multiseed, dim_in=dim_embedding, seed=seed_dicoba,
        hidden_dim=konfigurasi_multiseed["hidden_dim"], dropout=konfigurasi_multiseed["dropout"],
        lr=konfigurasi_multiseed["lr"], weight_decay=konfigurasi_multiseed["weight_decay"],
        label_smoothing=konfigurasi_multiseed["label_smoothing"], batch_size=konfigurasi_multiseed["batch_size"],
        n_layers=konfigurasi_multiseed.get("n_layers", 1), hidden_dim2=konfigurasi_multiseed.get("hidden_dim2"),
        activation=konfigurasi_multiseed.get("activation", "silu"), norm_type=konfigurasi_multiseed.get("norm_type", "rmsnorm"),
        loss_type=konfigurasi_multiseed["loss_type"], focal_gamma=konfigurasi_multiseed.get("focal_gamma"),
        skema_class_weight=konfigurasi_multiseed.get("skema_class_weight", "inverse_freq"),
        beta_effective_number=konfigurasi_multiseed.get("beta_effective_number", 0.999),
        bobot_manual_class_weight=konfigurasi_multiseed.get("bobot_manual_class_weight"),
        gunakan_sampler=konfigurasi_multiseed.get("gunakan_sampler", False),
        bobot_sampler_manual=konfigurasi_multiseed.get("bobot_sampler_manual"),
    )
    probs_seed = prediksi_probs_dengan_tta(model_seed, embedding_test_clean, embedding_test_tta)
    skor_seed_test = bandingkan_dengan_ground_truth(
        f"CE full data, seed {seed_dicoba}", torch.argmax(probs_seed, dim=1).cpu().numpy(),
        file_list_test, LABEL_TEST_PATH_BANDING,
    )
    daftar_model_multiseed.append(model_seed)
    daftar_probs_multiseed.append(probs_seed)

    path_model_seed = f"{MODEL_SAVE_DIR}/model_mlp_probe_siglip_fulldata_ce_seed{seed_dicoba}_{TIMESTAMP_RUN}.pth"
    torch.save(model_seed.state_dict(), path_model_seed)
    torch.cuda.empty_cache()

probs_multiseed_blend = torch.stack(daftar_probs_multiseed, dim=0).mean(dim=0)
prediksi_multiseed_idx = torch.argmax(probs_multiseed_blend, dim=1).cpu().numpy()

pd.DataFrame({
    "File": file_list_test, "Prediksi": [IDX_TO_LABEL[i] for i in prediksi_multiseed_idx],
}).to_csv(f"{ROOT_DIR}/submission_multiseed_ce_{TIMESTAMP_RUN}.csv", index=False)

f1_multiseed_test = bandingkan_dengan_ground_truth(
    "CE full data, multi seed ensemble", prediksi_multiseed_idx, file_list_test, LABEL_TEST_PATH_BANDING,
)

hasil_multiseed_ce = {
    "nama_seed": DAFTAR_SEED_MULTISEED,
    "model_list": daftar_model_multiseed,
    "probs": probs_multiseed_blend.cpu().numpy(),
    "prediksi_idx": prediksi_multiseed_idx,
    "f1_test": f1_multiseed_test,
}

print(f"\nMacro F1 test, multi seed ensemble ({len(DAFTAR_SEED_MULTISEED)} seed) CE full data, {f1_multiseed_test:.4f}")
print(f"Pembanding, model full data CE seed tunggal, {hasil_normal_ce['f1_test_fulldata_headnormal']:.4f}")

## Pilih Otomatis Metode Terbaik, Simpan submission.csv dan Model Pemenang

Semua metode yang sudah dicoba di atas dikumpulkan di sini: K-Fold Ensemble, Blend (K-Fold + Full Data), Head Normal (tanpa fold), Full Data (head normal), dan Blend (Head Normal + Full Data), masing masing untuk loss CrossEntropy dan Focal. Semuanya dibandingkan macro F1 test-nya ke ground truth (`LABEL_TEST_PATH_BANDING`).

Metode dengan F1 test tertinggi dipakai sebagai `submission.csv` final (tanpa suffix nama run/timestamp, supaya jelas ini yang dipakai), dan model (atau kumpulan model, kalau metodenya ensemble/blend) yang menghasilkan skor itu disalin ke folder terpisah `BEST_MODEL_DIR`, supaya tidak tercampur dengan model model percobaan lain di `MODEL_SAVE_DIR`.

In [ ]:
# ============================================================
# PILIH METODE TERBAIK DARI SEMUA PIPELINE (K-FOLD, BLEND, HEAD NORMAL,
# FULL DATA), BERDASARKAN MACRO F1 TEST TERHADAP GROUND TRUTH
# ============================================================
import json

kandidat_metode = []


def _daftar_model_fold_dict(daftar_model_fold, tag):
    return {f"model_fold{i}_{tag}": m for i, m in enumerate(daftar_model_fold)}


if hasil_ce is not None:
    kandidat_metode.append({
        "nama": "CE - K-Fold Ensemble",
        "f1_test": hasil_ce["f1_test_kfold"],
        "prediksi_idx": hasil_ce["prediksi_ensemble_idx"],
        "probs": hasil_ce["probs_ensemble"],
        "model": _daftar_model_fold_dict(hasil_ce["daftar_model_fold"], "ce"),
    })
    kandidat_metode.append({
        "nama": "CE - Blend (K-Fold + Full Data)",
        "f1_test": hasil_ce["f1_test_blend"],
        "prediksi_idx": hasil_ce["prediksi_blend_idx"],
        "probs": hasil_ce["probs_blend"],
        "model": {**_daftar_model_fold_dict(hasil_ce["daftar_model_fold"], "ce"), "model_fulldata_ce": hasil_ce["model_full"]},
    })

if hasil_focal is not None:
    kandidat_metode.append({
        "nama": "Focal - K-Fold Ensemble",
        "f1_test": hasil_focal["f1_test_kfold"],
        "prediksi_idx": hasil_focal["prediksi_ensemble_idx"],
        "probs": hasil_focal["probs_ensemble"],
        "model": _daftar_model_fold_dict(hasil_focal["daftar_model_fold"], "focal"),
    })
    kandidat_metode.append({
        "nama": "Focal - Blend (K-Fold + Full Data)",
        "f1_test": hasil_focal["f1_test_blend"],
        "prediksi_idx": hasil_focal["prediksi_blend_idx"],
        "probs": hasil_focal["probs_blend"],
        "model": {**_daftar_model_fold_dict(hasil_focal["daftar_model_fold"], "focal"), "model_fulldata_focal": hasil_focal["model_full"]},
    })

if hasil_normal_ce is not None:
    kandidat_metode.append({
        "nama": "CE - Head Normal (tanpa fold)",
        "f1_test": hasil_normal_ce["f1_test_headnormal"],
        "prediksi_idx": hasil_normal_ce["prediksi_normal_idx"],
        "probs": hasil_normal_ce["probs_normal"],
        "model": {"model_headnormal_ce": hasil_normal_ce["model_headnormal"]},
    })
    kandidat_metode.append({
        "nama": "CE - Full Data (head normal)",
        "f1_test": hasil_normal_ce["f1_test_fulldata_headnormal"],
        "prediksi_idx": hasil_normal_ce["prediksi_full_normal_idx"],
        "probs": hasil_normal_ce["probs_full_normal"],
        "model": {"model_fulldata_headnormal_ce": hasil_normal_ce["model_fulldata_headnormal"]},
    })
    kandidat_metode.append({
        "nama": "CE - Blend Head Normal + Full Data",
        "f1_test": hasil_normal_ce["f1_test_blend_headnormal"],
        "prediksi_idx": hasil_normal_ce["prediksi_blend_normal_idx"],
        "probs": hasil_normal_ce["probs_blend_normal"],
        "model": {
            "model_headnormal_ce": hasil_normal_ce["model_headnormal"],
            "model_fulldata_headnormal_ce": hasil_normal_ce["model_fulldata_headnormal"],
        },
    })

if hasil_normal_focal is not None:
    kandidat_metode.append({
        "nama": "Focal - Head Normal (tanpa fold)",
        "f1_test": hasil_normal_focal["f1_test_headnormal"],
        "prediksi_idx": hasil_normal_focal["prediksi_normal_idx"],
        "probs": hasil_normal_focal["probs_normal"],
        "model": {"model_headnormal_focal": hasil_normal_focal["model_headnormal"]},
    })
    kandidat_metode.append({
        "nama": "Focal - Full Data (head normal)",
        "f1_test": hasil_normal_focal["f1_test_fulldata_headnormal"],
        "prediksi_idx": hasil_normal_focal["prediksi_full_normal_idx"],
        "probs": hasil_normal_focal["probs_full_normal"],
        "model": {"model_fulldata_headnormal_focal": hasil_normal_focal["model_fulldata_headnormal"]},
    })
    kandidat_metode.append({
        "nama": "Focal - Blend Head Normal + Full Data",
        "f1_test": hasil_normal_focal["f1_test_blend_headnormal"],
        "prediksi_idx": hasil_normal_focal["prediksi_blend_normal_idx"],
        "probs": hasil_normal_focal["probs_blend_normal"],
        "model": {
            "model_headnormal_focal": hasil_normal_focal["model_headnormal"],
            "model_fulldata_headnormal_focal": hasil_normal_focal["model_fulldata_headnormal"],
        },
    })

if "hasil_multiseed_ce" in globals() and hasil_multiseed_ce is not None:
    kandidat_metode.append({
        "nama": f"CE - Multi Seed Ensemble ({len(hasil_multiseed_ce['nama_seed'])} seed, full data)",
        "f1_test": hasil_multiseed_ce["f1_test"],
        "prediksi_idx": hasil_multiseed_ce["prediksi_idx"],
        "probs": hasil_multiseed_ce["probs"],
        "model": {
            f"model_fulldata_ce_seed{s}": m
            for s, m in zip(hasil_multiseed_ce["nama_seed"], hasil_multiseed_ce["model_list"])
        },
    })

kandidat_valid = [k for k in kandidat_metode if k["f1_test"] is not None]
assert len(kandidat_valid) > 0, "Tidak ada metode dengan skor F1 test valid, cek LABEL_TEST_PATH_BANDING ada isinya atau tidak"

df_ringkasan_semua_metode = pd.DataFrame([
    {"metode": k["nama"], "f1_test": k["f1_test"]} for k in kandidat_valid
]).sort_values("f1_test", ascending=False).reset_index(drop=True)

print("Ringkasan macro F1 test, semua metode, terurut dari yang tertinggi")
print(df_ringkasan_semua_metode.to_string(index=False))

kandidat_terbaik = max(kandidat_valid, key=lambda k: k["f1_test"])
print(f"\nMetode dengan F1 test tertinggi, {kandidat_terbaik['nama']} (F1 {kandidat_terbaik['f1_test']:.4f})")

# ---- Simpan submission.csv final, dari metode terbaik ----
df_submission_final = pd.DataFrame({
    "File": file_list_test,
    "Prediksi": [IDX_TO_LABEL[i] for i in kandidat_terbaik["prediksi_idx"]],
})
PATH_SUBMISSION_FINAL = f"{ROOT_DIR}/submission.csv"
df_submission_final.to_csv(PATH_SUBMISSION_FINAL, index=False)
print(f"submission.csv disimpan di {PATH_SUBMISSION_FINAL}")

# ---- Simpan model pemenang ke folder terpisah, biar tidak campur dengan model percobaan lain ----
BEST_MODEL_DIR = f"{ROOT_DIR}/Train_model/Best_model/overall_terbaik_{TAG_VERSI_SIGLIP}"
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

# Bersihkan file .pth lama di folder ini dulu, supaya tidak ada model dari
# run sebelumnya yang ketinggalan dan bikin bingung mana yang benar benar dipakai.
for f_lama in glob.glob(f"{BEST_MODEL_DIR}/*.pth"):
    os.remove(f_lama)

for nama_model, model_obj in kandidat_terbaik["model"].items():
    torch.save(model_obj.state_dict(), f"{BEST_MODEL_DIR}/{nama_model}.pth")

info_model_terbaik = {
    "metode_terbaik": kandidat_terbaik["nama"],
    "f1_test": float(kandidat_terbaik["f1_test"]),
    "file_model": sorted(kandidat_terbaik["model"].keys()),
    "timestamp_run": TIMESTAMP_RUN,
    "path_submission": PATH_SUBMISSION_FINAL,
}
with open(f"{BEST_MODEL_DIR}/info_model_terbaik.json", "w") as f_info:
    json.dump(info_model_terbaik, f_info, indent=2)

print(f"Model metode terbaik ({len(kandidat_terbaik['model'])} file) disimpan di {BEST_MODEL_DIR}")
print(f"Detail tersimpan di {BEST_MODEL_DIR}/info_model_terbaik.json")


## Tetangga Terdekat di Train dan Test, Untuk Gambar yang Salah di Metode Juara

Bagian visualisasi kesalahan per metode di atas sudah mencakup semua metode yang diuji. Di sini fokus ke satu metode saja, yaitu metode juara (`kandidat_terbaik`, hasil pemilihan otomatis di atas), gambar test yang salah menurut metode itu dicari tetangga terdekatnya di embedding test dan train (jarak cosine), untuk mengecek kecurigaan ada label yang salah di train.

In [ ]:
# ============================================================
# TETANGGA TERDEKAT DI EMBEDDING TEST DAN TRAIN, GAMBAR SALAH METODE JUARA
# ============================================================
# Sengaja dibuat berdiri sendiri (tidak bergantung ke cell visualisasi
# semua metode di bawah), supaya urutan jalannya bisa taruh visualisasi
# semua metode di paling akhir notebook.
from sklearn.metrics.pairwise import cosine_similarity


def muat_ground_truth_test(path_label_test):
    df_gt = pd.read_csv(path_label_test, encoding="utf-8-sig", sep=None, engine="python")
    df_gt.columns = [c.strip() for c in df_gt.columns]
    if "id" in df_gt.columns:
        df_gt["id"] = pd.to_numeric(df_gt["id"].astype(str).str.strip(), errors="coerce")
    elif "file" in df_gt.columns:
        df_gt["id"] = pd.to_numeric(
            df_gt["file"].astype(str).str.strip().apply(lambda x: os.path.splitext(x)[0]), errors="coerce"
        )
    else:
        raise ValueError(f"Ground truth {path_label_test} harus punya kolom 'id' atau 'file'")
    df_gt["label"] = pd.to_numeric(df_gt["label"].astype(str).str.strip(), errors="coerce")
    baris_valid = df_gt["label"].isin([0, 1, 2]) & df_gt["id"].notna()
    df_gt_bersih_lokal = df_gt.loc[baris_valid, ["id", "label"]].copy()
    df_gt_bersih_lokal["id"] = df_gt_bersih_lokal["id"].astype(int)
    df_gt_bersih_lokal["label"] = df_gt_bersih_lokal["label"].astype(int)
    return df_gt_bersih_lokal


df_gt_bersih_juara = muat_ground_truth_test(LABEL_TEST_PATH_BANDING)
daftar_id_test_juara = [int(os.path.splitext(f)[0]) for f in file_list_test]

probs_juara_np = kandidat_terbaik["probs"]
prediksi_juara_idx = kandidat_terbaik["prediksi_idx"]

df_eval_juara = pd.DataFrame({
    "id": daftar_id_test_juara,
    "file": file_list_test,
    "prediksi_idx": prediksi_juara_idx,
    "prob_recyclable": probs_juara_np[:, LABEL_MAP["Recyclable"]],
    "prob_electronic": probs_juara_np[:, LABEL_MAP["Electronic"]],
    "prob_organic": probs_juara_np[:, LABEL_MAP["Organic"]],
})
df_eval_juara = df_eval_juara.merge(df_gt_bersih_juara, on="id", how="inner").rename(columns={"label": "label_asli_idx"})

matriks_prob_juara = df_eval_juara[["prob_recyclable", "prob_electronic", "prob_organic"]].values
df_eval_juara["prediksi_label"] = df_eval_juara["prediksi_idx"].map(IDX_TO_LABEL)
df_eval_juara["label_asli"] = df_eval_juara["label_asli_idx"].map(IDX_TO_LABEL)
df_eval_juara["prob_prediksi"] = matriks_prob_juara[np.arange(len(df_eval_juara)), df_eval_juara["prediksi_idx"].values]
df_eval_juara["prob_label_asli"] = matriks_prob_juara[np.arange(len(df_eval_juara)), df_eval_juara["label_asli_idx"].values]
df_eval_juara["benar"] = df_eval_juara["prediksi_idx"] == df_eval_juara["label_asli_idx"]

df_salah_juara = df_eval_juara[~df_eval_juara["benar"]].reset_index(drop=True)
print(f"Metode juara, {kandidat_terbaik['nama']}, jumlah gambar test yang salah, {len(df_salah_juara)}")

label_test_gt_map = dict(zip(df_gt_bersih_juara["id"], df_gt_bersih_juara["label"]))
label_test_gt_arr = np.array([
    label_test_gt_map.get(int(os.path.splitext(f)[0]), -1) for f in file_list_test
])

path_test_per_baris = np.array([os.path.join(TEST_DIR_DRIVE, f) for f in file_list_test])
path_train_per_baris = df_train[KOLOM_PATH_MANIFEST].to_numpy()[groups_train]

K_TETANGGA = 5


def cari_tetangga_terdekat(vektor_query, embedding_kandidat, k, label_kandidat, path_kandidat, indeks_kecuali=None):
    sim = cosine_similarity(vektor_query.reshape(1, -1), embedding_kandidat)[0]
    jarak = 1.0 - sim
    urutan = np.argsort(jarak)
    if indeks_kecuali is not None:
        urutan = urutan[urutan != indeks_kecuali]
    urutan_top = urutan[:k]
    return pd.DataFrame({
        "jarak_cosine": jarak[urutan_top],
        "label": [IDX_TO_LABEL.get(int(l), "tidak diketahui") for l in label_kandidat[urutan_top]],
        "path": path_kandidat[urutan_top],
    })


if len(df_salah_juara) == 0:
    print("Tidak ada kesalahan prediksi di metode juara, pencarian tetangga terdekat dilewati")
else:
    for i, baris in df_salah_juara.iterrows():
        idx_test_asli = file_list_test.index(baris["file"])
        vektor_query = embedding_test_clean[idx_test_asli]

        print(f"\n{'=' * 70}")
        print(f"Test id {baris['id']} ({baris['file']}), label asli {baris['label_asli']}, diprediksi {baris['prediksi_label']}")
        print(f"Probabilitas ke label asli {baris['prob_label_asli']:.4f}, ke prediksi {baris['prob_prediksi']:.4f}")

        print(f"\nTop {K_TETANGGA} tetangga terdekat di TEST")
        tetangga_test = cari_tetangga_terdekat(
            vektor_query, embedding_test_clean, K_TETANGGA,
            label_test_gt_arr, path_test_per_baris, indeks_kecuali=idx_test_asli,
        )
        print(tetangga_test.to_string(index=False))

        print(f"\nTop {K_TETANGGA} tetangga terdekat di TRAIN (mencurigakan kalau labelnya beda dari label asli test ini)")
        tetangga_train = cari_tetangga_terdekat(
            vektor_query, embedding_train, K_TETANGGA, label_train, path_train_per_baris,
        )
        tetangga_train["mencurigakan"] = tetangga_train["label"] != baris["label_asli"]
        print(tetangga_train.to_string(index=False))

## Visualisasi Kesalahan terhadap Ground Truth, Untuk Setiap Metode

Generalisasi dari analisis kesalahan di atas (yang sebelumnya cuma untuk satu model, CE Full Data), sekarang dijalankan untuk **semua kandidat metode** yang ada di `kandidat_metode` (CE/Focal x K-Fold, Blend, Head Normal, Full Data, Blend Head Normal+Full).

Untuk tiap metode, ditampilkan macro F1 test hasil hitung ulang, tabel sampel yang salah beserta gap probabilitas, dan grid gambar sampel yang salah (sama persis formatnya seperti visualisasi single-method sebelumnya).

Di akhir ada tabel ringkasan konsistensi, id test mana saja yang salah dan di berapa banyak metode. Kalau satu id konsisten salah di hampir semua metode, itu sinyal datanya memang ambigu/berat sebelah. Kalau cuma salah di satu dua metode, itu lebih mengarah ke noise dari training run tertentu (bukan masalah datanya).

In [ ]:
# ============================================================
# VISUALISASI KESALAHAN TERHADAP GROUND TRUTH, UNTUK SETIAP METODE
# ============================================================
import matplotlib.pyplot as plt


def muat_ground_truth_test(path_label_test):
    df_gt = pd.read_csv(path_label_test, encoding="utf-8-sig", sep=None, engine="python")
    df_gt.columns = [c.strip() for c in df_gt.columns]
    if "id" in df_gt.columns:
        df_gt["id"] = pd.to_numeric(df_gt["id"].astype(str).str.strip(), errors="coerce")
    elif "file" in df_gt.columns:
        df_gt["id"] = pd.to_numeric(
            df_gt["file"].astype(str).str.strip().apply(lambda x: os.path.splitext(x)[0]), errors="coerce"
        )
    else:
        raise ValueError(f"Ground truth {path_label_test} harus punya kolom 'id' atau 'file'")
    df_gt["label"] = pd.to_numeric(df_gt["label"].astype(str).str.strip(), errors="coerce")
    baris_valid = df_gt["label"].isin([0, 1, 2]) & df_gt["id"].notna()
    df_gt_bersih_lokal = df_gt.loc[baris_valid, ["id", "label"]].copy()
    df_gt_bersih_lokal["id"] = df_gt_bersih_lokal["id"].astype(int)
    df_gt_bersih_lokal["label"] = df_gt_bersih_lokal["label"].astype(int)
    return df_gt_bersih_lokal


df_gt_bersih_semua = muat_ground_truth_test(LABEL_TEST_PATH_BANDING)
daftar_id_test = [int(os.path.splitext(f)[0]) for f in file_list_test]


def analisis_kesalahan_metode(nama_metode, probs_np, tampilkan_grafik=True):
    prediksi_idx = probs_np.argmax(axis=1)

    df_eval = pd.DataFrame({
        "id": daftar_id_test,
        "file": file_list_test,
        "prediksi_idx": prediksi_idx,
        "prob_recyclable": probs_np[:, LABEL_MAP["Recyclable"]],
        "prob_electronic": probs_np[:, LABEL_MAP["Electronic"]],
        "prob_organic": probs_np[:, LABEL_MAP["Organic"]],
    })
    df_eval = df_eval.merge(df_gt_bersih_semua, on="id", how="inner").rename(columns={"label": "label_asli_idx"})

    matriks_prob = df_eval[["prob_recyclable", "prob_electronic", "prob_organic"]].values
    df_eval["prediksi_label"] = df_eval["prediksi_idx"].map(IDX_TO_LABEL)
    df_eval["label_asli"] = df_eval["label_asli_idx"].map(IDX_TO_LABEL)
    df_eval["prob_prediksi"] = matriks_prob[np.arange(len(df_eval)), df_eval["prediksi_idx"].values]
    df_eval["prob_label_asli"] = matriks_prob[np.arange(len(df_eval)), df_eval["label_asli_idx"].values]
    df_eval["gap_probabilitas"] = df_eval["prob_prediksi"] - df_eval["prob_label_asli"]
    df_eval["benar"] = df_eval["prediksi_idx"] == df_eval["label_asli_idx"]

    f1_metode = f1_score(df_eval["label_asli_idx"], df_eval["prediksi_idx"], average="macro")
    print(f"\n{'=' * 70}\n[{nama_metode}] Macro F1 test hasil hitung ulang, {f1_metode:.4f}")

    df_salah = df_eval[~df_eval["benar"]].sort_values("gap_probabilitas", ascending=False).reset_index(drop=True)
    n_salah = len(df_salah)
    print(f"[{nama_metode}] Jumlah sampel test yang salah, {n_salah} dari {len(df_eval)}")

    if n_salah > 0:
        print(df_salah[
            ["id", "file", "label_asli", "prediksi_label", "prob_label_asli", "prob_prediksi", "gap_probabilitas"]
        ].to_string(index=False))

    if tampilkan_grafik and n_salah > 0:
        n_kolom = min(4, n_salah)
        n_baris_grid = int(np.ceil(n_salah / n_kolom))
        fig, axes = plt.subplots(n_baris_grid, n_kolom, figsize=(4 * n_kolom, 4.5 * n_baris_grid))
        axes = np.array(axes).reshape(-1)
        fig.suptitle(f"{nama_metode}, macro F1 test {f1_metode:.4f}", fontsize=13, y=1.02)
        for i, baris in df_salah.iterrows():
            img = baca_gambar_aman(os.path.join(TEST_DIR_DRIVE, baris["file"]))
            axes[i].imshow(img)
            axes[i].axis("off")
            axes[i].set_title(
                f"id {baris['id']}\n"
                f"asli: {baris['label_asli']} (p={baris['prob_label_asli']:.3f})\n"
                f"prediksi: {baris['prediksi_label']} (p={baris['prob_prediksi']:.3f})",
                fontsize=9, color="crimson",
            )
        for j in range(n_salah, len(axes)):
            axes[j].axis("off")
        plt.tight_layout()
        plt.show()

    return df_salah.assign(metode=nama_metode, f1_test_hitung_ulang=f1_metode)


daftar_df_salah_semua_metode = []
for k in kandidat_valid:
    df_salah_k = analisis_kesalahan_metode(k["nama"], k["probs"])
    daftar_df_salah_semua_metode.append(df_salah_k)

df_salah_gabungan = pd.concat(daftar_df_salah_semua_metode, ignore_index=True) if daftar_df_salah_semua_metode else pd.DataFrame()

if len(df_salah_gabungan) == 0:
    print("\nTidak ada satupun sampel test yang salah di semua metode")
else:
    ringkasan_konsistensi = (
        df_salah_gabungan.groupby(["id", "file", "label_asli"])
        .agg(
            jumlah_metode_salah=("metode", "nunique"),
            metode_yang_salah=("metode", lambda s: ", ".join(sorted(s.unique()))),
        )
        .sort_values("jumlah_metode_salah", ascending=False)
    )
    print(f"\n{'=' * 70}\nRINGKASAN KONSISTENSI KESALAHAN ANTAR METODE")
    print(f"Total metode yang dianalisis, {len(kandidat_valid)}")
    print(
        "id test yang konsisten salah di banyak metode dicurigai memang ambigu/data berat sebelah.\n"
        "id test yang cuma salah di 1-2 metode dicurigai noise dari training run tertentu, bukan masalah datanya.\n"
    )
    print(ringkasan_konsistensi.to_string())
